# EDA Star Wars Business Intelligence

Objetivo: explorar, limpiar y preparar dos datasets relacionados con Star Wars para construir un dashboard ejecutivo en Power BI.

## 1. Importacion de librerias


En esta sección se importan las librerías necesarias para trabajar con los datos.

También se configuran algunas opciones de visualización de pandas para poder ver más columnas y filas en el notebook.

Además, se crea la carpeta `data/clean/`, donde se guardarán los datasets limpios generados durante el proceso.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re
import os


PROJECT_MARKERS = ["README.md", "requirements.txt", "data"]
BASE_DIR = Path.cwd()

while BASE_DIR.parent != BASE_DIR and not all((BASE_DIR / marker).exists() for marker in PROJECT_MARKERS):
    BASE_DIR = BASE_DIR.parent

if not all((BASE_DIR / marker).exists() for marker in PROJECT_MARKERS):
    raise FileNotFoundError("No se encontro la raiz del proyecto DashboardStarWars")

os.chdir(BASE_DIR)

print("Ahora Python está en:")
print(Path.cwd())

'''Esta celda sirve para comprobar que archicos puede ver python desde donde esta
 Esto es importante para saber cómo referenciar los archivos que queremos usar en el notebook.
 Si el notebook está en la misma carpeta que los archivos, 
 entonces podemos referenciarlos directamente por su nombre.
 Si el notebook está en una carpeta diferente, entonces tenemos que usar rutas relativas o absolutas para referenciarlos.
'''



pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


Ahora Python está en:
C:\Users\elena\OneDrive\Documentos\BOOTCAMP IA\DashboardStarWars


## 1.2 Carga de datos

Coloca los CSV originales en `data/raw` y ajusta los nombres de archivo si hace falta.

In [2]:
RAW_DIR = Path("data/raw") #definimos la carpeta base donde se encuentran los datos en bruto, 


# Creamos la ruta completa al CSV principal de la encuesta de Star Wars.
STARWARS_PATH = RAW_DIR / "StarWars" / "StarWars.csv" 


# Creamos la ruta a la carpeta donde están los CSV extra descargados de Kaggle.
KAGGLE_CSV_DIR = RAW_DIR / "starwars_kaggle" / "archive (1)" / "csv"
FILMS_BUSINESS_PATH = RAW_DIR / "films_business.csv"

print("Ruta StarWars:", STARWARS_PATH) #mostramos por pantalla la ruta del archivo principal
print("Existe StarWars:", STARWARS_PATH.exists()) #devuelve True si el archivo existe, False si no

print("Ruta Kaggle CSV:", KAGGLE_CSV_DIR) 
print("Existe carpeta Kaggle CSV:", KAGGLE_CSV_DIR.exists()) #comprobamos si existe o no


#comprobamos si existe una archivo en especifico

print("Existe characters.csv:", (KAGGLE_CSV_DIR / "characters.csv").exists()) 
print("Existe films_business.csv:", FILMS_BUSINESS_PATH.exists())

#pasamos a leer con pandas los archivos csv de las database que hemos elegido,
#el resultado se guarda en un dataframe para cada uno de los archivos
df_starwars_raw= pd.read_csv(STARWARS_PATH)

df_characters = pd.read_csv(KAGGLE_CSV_DIR / "characters.csv")
df_films = pd.read_csv(KAGGLE_CSV_DIR / "films.csv")
df_planets = pd.read_csv(KAGGLE_CSV_DIR / "planets.csv")
df_species = pd.read_csv(KAGGLE_CSV_DIR / "species.csv")
df_starships = pd.read_csv(KAGGLE_CSV_DIR / "starships.csv")
df_vehicles = pd.read_csv(KAGGLE_CSV_DIR / "vehicles.csv")
df_quotes = pd.read_csv(KAGGLE_CSV_DIR / "quotes.csv")
df_weapons = pd.read_csv(KAGGLE_CSV_DIR / "weapons.csv")
df_droids = pd.read_csv(KAGGLE_CSV_DIR / "droids.csv")
df_films_business = pd.read_csv(FILMS_BUSINESS_PATH)


print("\nDatasets cargados correctamente:")

#mostramos los tamaños de los datasets cargados, el número de filas y columnas de cada uno

print("Characters:", df_characters.shape)
print("Films:", df_films.shape)
print("Planets:", df_planets.shape)
print("Species:", df_species.shape)
print("Starships:", df_starships.shape)
print("Vehicles:", df_vehicles.shape)
print("Quotes:", df_quotes.shape)
print("Weapons:", df_weapons.shape)
print("Droids:", df_droids.shape)
print("Films business:", df_films_business.shape)
#mostramos el tamaño del dataset principal de la encuesta de star wars 
#y las primeras filas para comprobar que se ha cargado correctamente
print("Dimensiones originales:", df_starwars_raw.shape)
display(df_starwars_raw.head())


Ruta StarWars: data\raw\StarWars\StarWars.csv
Existe StarWars: True
Ruta Kaggle CSV: data\raw\starwars_kaggle\archive (1)\csv
Existe carpeta Kaggle CSV: True
Existe characters.csv: True
Existe films_business.csv: True

Datasets cargados correctamente:
Characters: (112, 13)
Films: (11, 6)
Planets: (26, 12)
Species: (39, 11)
Starships: (56, 16)
Vehicles: (21, 14)
Quotes: (89, 4)
Weapons: (57, 9)
Droids: (14, 10)
Films business: (12, 15)
Dimensiones originales: (1187, 40)


,respondentid,have you seen any of the 6 films in the star wars franchise?,do you consider yourself to be a fan of the star wars film franchise?,which of the following star wars films have you seen? please select all that apply.,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,please rank the star wars films in order of preference with 1 being your favorite film in the franchise and 6 being your least favorite film.,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,please state whether you view the following characters favorably,unfavorably,or are unfamiliar with him/her.,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,which character shot first?,are you familiar with the expanded universe?,do you consider yourself to be a fan of the expanded universe?,do you consider yourself to be a fan of the star trek franchise?,gender,age,household income,education,location (census region)
0,NaN,response,response,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,star wars: episode iv a new hope,star wars: episode v the empire strikes back,star wars: episode vi return of the jedi,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,star wars: episode iv a new hope,star wars: episode v the empire strikes back,star wars: episode vi return of the jedi,han solo,luke skywalker,princess leia organa,anakin skywalker,obi wan kenobi,emperor palpatine,darth vader,lando calrissian,boba fett,c-3p0,r2 d2,jar jar binks,padme amidala,yoda,response,response,response,response,response,response,response,response,response,NaN,NaN
1,3.292880e+09,yes,yes,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,star wars: episode iv a new hope,star wars: episode v the empire strikes back,star wars: episode vi return of the jedi,3,2,1,4,5,6,very favorably,very favorably,very favorably,very favorably,very favorably,very favorably,very favorably,unfamiliar (n/a),unfamiliar (n/a),very favorably,very favorably,very favorably,very favorably,very favorably,i don't understand this question,yes,no,no,male,18-29,NaN,high school degree,south atlantic,NaN,NaN
2,3.292880e+09,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,male,18-29,$0 - $24,999,bachelor degree,west south central,NaN
3,3.292765e+09,yes,no,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,NaN,NaN,NaN,1,2,3,4,5,6,somewhat favorably,somewhat favorably,somewhat favorably,somewhat favorably,somewhat favorably,unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),i don't understand this question,no,NaN,no,male,18-29,$0 - $24,999,high school degree,west north central,NaN
4,3.292763e+09,yes,yes,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,star wars: episode iv a new hope,star wars: episode v the empire strikes back,star wars: episode vi return of the jedi,5,6,1,2,4,3,very favorably,very favorably,very favorably,very favorably,very favorably,somewhat favorably,very favorably,somewhat favorably,somewhat unfavorably,very favorably,very favorably,very favorably,very favorably,very favorably,i don't understand this question,no,NaN,yes,male,18-29,$100,000 - $149,999,some college or associate degree,west north central


## x. Funciones generales de limpieza

Antes de limpiar cada dataset, se definen funciones reutilizables. Estas funciones permiten:

- Normalizar los nombres de las columnas a un formato común en inglés y `snake_case`.
- Aplicar diccionarios de equivalencias para que una misma variable tenga el mismo nombre en todos los datasets.
- Limpiar valores de texto vacíos o no informativos.
- Reutilizar la misma lógica en los datasets 


In [3]:

# FUNCIONES GENERALES DE LIMPIEZA


def normalize_column_names(df):
    """
    Normaliza nombres de columnas a snake_case básico.

    Ejemplo:
    'Character Name' -> 'character_name'
    'Año de aparición' -> 'ano_de_aparicion'
    """
    df = df.copy()

    normalized_columns = []

    for column in df.columns:
        # Convertimos el nombre de la columna a texto,
        # quitamos espacios al principio/final y pasamos a minúsculas.
        column_name = str(column).strip().lower()

        # Quitamos acentos y la ñ.
        accent_map = str.maketrans("áéíóúüñ", "aeiouun")
        column_name = column_name.translate(accent_map)

        # Sustituimos cualquier carácter que no sea letra o número por "_".
        column_name = re.sub(r"[^a-z0-9]+", "_", column_name)

        # Si hay varios "_" seguidos, los dejamos en uno solo.
        # También quitamos "_" al principio o al final.
        column_name = re.sub(r"_+", "_", column_name).strip("_")

        normalized_columns.append(column_name)

    df.columns = normalized_columns

    return df


def clean_text(value):
    """
    Limpia un valor de texto individual:
    - si ya es nulo, lo deja como NaN
    - quita espacios al principio y al final
    - convierte textos vacíos o falsos nulos en NaN
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value == "" or value.lower() in ["nan", "none", "null"]:
        return np.nan

    return value


def clean_text_columns(df):
    """
    Limpia todas las columnas de texto de un DataFrame.
    """
    df = df.copy()

    text_columns = df.select_dtypes(include=["object", "string"]).columns

    for column in text_columns:
        df[column] = df[column].apply(clean_text)

    return df


def apply_column_mapping(df, column_mapping):
    """
    Renombra columnas usando un diccionario de equivalencias.

    Ejemplo:
    {"name": "character_name"}
    """
    df = df.copy()

    existing_mapping = {
        source_column: target_column
        for source_column, target_column in column_mapping.items()
        if source_column in df.columns
    }

    return df.rename(columns=existing_mapping)


def add_missing_columns(df, required_columns):
    """
    Añade columnas necesarias que no existan en el dataset.

    Las columnas nuevas se rellenan con NaN.
    """
    df = df.copy()

    for column in required_columns:
        if column not in df.columns:
            df[column] = np.nan

    return df

# 2. Estudio y analisis datasec/encuesta Starwars


Antes de limpiar o analizar el dataset de la encuesta de Star Wars, realizamos una revisión estructural porque detectamos que no se comportaba como un CSV tradicional.

Al cargar el archivo observamos que muchas columnas aparecían con nombres automáticos como `unnamed_4`, `unnamed_5`, `unnamed_18`, etc. Esto no significaba necesariamente que fueran columnas inútiles o vacías, sino que formaban parte de preguntas agrupadas. Es decir, algunas preguntas de la encuesta ocupaban varias columnas, una por cada opción de respuesta.

También detectamos que la fila 0 del dataset no era una respuesta real de una persona encuestada. Esa primera fila contenía información auxiliar o metadata sobre las opciones de algunas preguntas. Por ejemplo, en las columnas de películas vistas y ranking, la fila 0 indicaba a qué episodio correspondía cada columna. En las columnas de opinión sobre personajes, la fila 0 permitía identificar personajes como Anakin Skywalker, Obi Wan Kenobi, Darth Vader, Yoda, etc.

Por este motivo, tomamos la decisión de separar el dataset en dos partes:

- `df_starwars_raw`: copia original del dataset sin modificar.
- `metadata_row`: primera fila del dataset, guardada como referencia para interpretar columnas.
- `df_survey`: versión de trabajo de la encuesta, eliminando la fila 0 para quedarnos solo con respuestas reales.

Después creamos un diccionario o mapa de columnas (`column_dictionary`) para comparar la posición de cada columna, su nombre original, el valor de la fila 0 y el porcentaje de valores nulos. Este paso nos permitió entender qué representaba cada columna antes de aplicar transformaciones.

### Conclusiones del análisis estructural

Tras revisar columnas, valores únicos y la fila 0, concluimos que la encuesta está organizada en varios bloques:

1. **Preguntas generales iniciales**  
   Incluyen si la persona ha visto alguna película de Star Wars y si se considera fan de la franquicia.

2. **Películas vistas**  
   Corresponden a las columnas 3 a 8. Aunque algunas aparecen como `unnamed`, la fila 0 indica qué película representa cada columna. En estas columnas, si aparece el nombre de la película, significa que la persona la ha visto; si aparece un valor nulo, significa que no la ha seleccionado.

3. **Ranking de películas**  
   Corresponde a las columnas 9 a 14. Estas columnas contienen valores del 1 al 6, donde 1 representa la película favorita y 6 la menos favorita. Por tanto, no deben tratarse como texto normal, sino como variables numéricas de ranking.

4. **Opinión sobre personajes**  
   Corresponde a las columnas 15 a 28. Estas columnas contienen respuestas como `very favorably`, `somewhat favorably`, `neutral`, `somewhat unfavorably`, `very unfavorably` o `unfamiliar`. Los nombres reales de los personajes se obtienen a partir de la fila 0.

5. **Preguntas finales y datos demográficos**  
   A partir de la columna 29 detectamos un desplazamiento entre el nombre de la columna y el contenido real. Por ejemplo, la columna `unnamed_29` contenía respuestas como `han`, `greedo` o `i don't understand this question`, por lo que decidimos interpretarla como la pregunta `which_character_shot_first`.

### Decisiones tomadas

Debido a esta estructura especial, decidimos no aplicar una limpieza automática directamente sobre todo el dataset original. En su lugar, seguimos este proceso:

1. Conservar una copia original del dataset.
2. Guardar la fila 0 como metadata.
3. Eliminar la fila 0 de la versión de trabajo.
4. Crear un mapa de columnas para interpretar las columnas `unnamed`.
5. Agrupar las columnas por bloques temáticos.
6. Renombrar manualmente las columnas importantes usando la información de la fila 0 y los valores únicos observados.
7. Analizar cada bloque según su naturaleza:
   - preguntas sí/no como variables categóricas,
   - películas vistas como selección múltiple,
   - ranking de películas como valores numéricos,
   - opiniones de personajes como variables ordinales/categóricas,
   - datos demográficos revisados con especial cuidado por el desplazamiento detectado.

Esta revisión previa es necesaria para evitar errores de interpretación. Si hubiéramos tratado el dataset como una tabla normal desde el principio, podríamos haber analizado columnas `unnamed` sin saber qué representaban realmente o haber usado nombres de columnas que no coincidían con el contenido real.

In [4]:

# 1. INSPECCIÓN INICIAL DE LA ESTRUCTURA


print("Columnas del dataset original:\n")

for i, col in enumerate(df_starwars_raw.columns):
    print(i, "->", col)



Columnas del dataset original:

0 -> respondentid
1 -> have you seen any of the 6 films in the star wars franchise?
2 -> do you consider yourself to be a fan of the star wars film franchise?
3 -> which of the following star wars films have you seen? please select all that apply.
4 -> Unnamed: 4
5 -> Unnamed: 5
6 -> Unnamed: 6
7 -> Unnamed: 7
8 -> Unnamed: 8
9 -> please rank the star wars films in order of preference with 1 being your favorite film in the franchise and 6 being your least favorite film.
10 -> Unnamed: 10
11 -> Unnamed: 11
12 -> Unnamed: 12
13 -> Unnamed: 13
14 -> Unnamed: 14
15 -> please state whether you view the following characters favorably
16 ->  unfavorably
17 ->  or are unfamiliar with him/her.
18 -> Unnamed: 18
19 -> Unnamed: 19
20 -> Unnamed: 20
21 -> Unnamed: 21
22 -> Unnamed: 22
23 -> Unnamed: 23
24 -> Unnamed: 24
25 -> Unnamed: 25
26 -> Unnamed: 26
27 -> Unnamed: 27
28 -> Unnamed: 28
29 -> Unnamed: 29
30 -> Unnamed: 30
31 -> which character shot first?
32 -> 

In [5]:

# INSPECCIÓN ESPECIAL DEL DATASET DE ENCUESTA STAR WARS


# Guardamos una copia del dataset original de la encuesta.
# Así no perdemos nunca la estructura tal como venía en el CSV.

df_survey = df_starwars_raw.copy()

# Guardamos la primera fila porque parece contener información de opciones/subpreguntas.
# No parece una respuesta real de una persona.

metadata_row = df_starwars_raw.iloc[0]

# Creamos una versión de trabajo de la encuesta quitando la fila 0.
# Esta será la tabla con respuestas reales.

df_survey = df_starwars_raw.drop(index=0).reset_index(drop=True)
print("Dimensiones originales:", df_starwars_raw.shape)
print("Dimensiones encuesta limpia de filas:", df_survey.shape)

display(df_survey.head())



Dimensiones originales: (1187, 40)
Dimensiones encuesta limpia de filas: (1186, 40)


,respondentid,have you seen any of the 6 films in the star wars franchise?,do you consider yourself to be a fan of the star wars film franchise?,which of the following star wars films have you seen? please select all that apply.,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,please rank the star wars films in order of preference with 1 being your favorite film in the franchise and 6 being your least favorite film.,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,please state whether you view the following characters favorably,unfavorably,or are unfamiliar with him/her.,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,which character shot first?,are you familiar with the expanded universe?,do you consider yourself to be a fan of the expanded universe?,do you consider yourself to be a fan of the star trek franchise?,gender,age,household income,education,location (census region)
0,3.292880e+09,yes,yes,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,star wars: episode iv a new hope,star wars: episode v the empire strikes back,star wars: episode vi return of the jedi,3,2,1,4,5,6,very favorably,very favorably,very favorably,very favorably,very favorably,very favorably,very favorably,unfamiliar (n/a),unfamiliar (n/a),very favorably,very favorably,very favorably,very favorably,very favorably,i don't understand this question,yes,no,no,male,18-29,NaN,high school degree,south atlantic,NaN,NaN
1,3.292880e+09,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,male,18-29,$0 - $24,999,bachelor degree,west south central,NaN
2,3.292765e+09,yes,no,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,NaN,NaN,NaN,1,2,3,4,5,6,somewhat favorably,somewhat favorably,somewhat favorably,somewhat favorably,somewhat favorably,unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),i don't understand this question,no,NaN,no,male,18-29,$0 - $24,999,high school degree,west north central,NaN
3,3.292763e+09,yes,yes,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,star wars: episode iv a new hope,star wars: episode v the empire strikes back,star wars: episode vi return of the jedi,5,6,1,2,4,3,very favorably,very favorably,very favorably,very favorably,very favorably,somewhat favorably,very favorably,somewhat favorably,somewhat unfavorably,very favorably,very favorably,very favorably,very favorably,very favorably,i don't understand this question,no,NaN,yes,male,18-29,$100,000 - $149,999,some college or associate degree,west north central
4,3.292731e+09,yes,yes,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,star wars: episode iv a new hope,star wars: episode v the empire strikes back,star wars: episode vi return of the jedi,5,4,6,2,1,3,very favorably,somewhat favorably,somewhat favorably,somewhat unfavorably,very favorably,very unfavorably,somewhat favorably,neither favorably nor unfavorably (neutral),very favorably,somewhat favorably,somewhat favorably,very unfavorably,somewhat favorably,somewhat favorably,greedo,yes,no,no,male,18-29,$100,000 - $149,999,some college or associate degree,west north central


In [6]:

# REPORTE DE NULOS DE LA ENCUESTA STAR WARS


def missing_report(df):
    """
    Crea un informe de valores nulos por columna.
    Muestra:
    - número de nulos
    - porcentaje de nulos
    Solo muestra columnas que tienen al menos un nulo.
    """
    return (
        pd.DataFrame({
            "nulos": df.isna().sum(),
            "porcentaje": (df.isna().mean() * 100).round(2),
        })
        .query("nulos > 0")
        .sort_values("porcentaje", ascending=False)
    )


# Aplicamos el reporte solo a la encuesta limpia, sin la fila 0 de metadata.
missing_survey = missing_report(df_survey)

display(missing_survey)

,nulos,porcentaje
which character shot first?,973,82.04
Unnamed: 5,636,53.63
Unnamed: 4,615,51.85
Unnamed: 6,579,48.82
location (census region),561,47.30
which of the following star wars films have you seen? please select all that apply.,513,43.25
Unnamed: 8,448,37.77
Unnamed: 7,428,36.09
Unnamed: 23,374,31.53
Unnamed: 20,372,31.37


In [7]:

# 4. MAPA DE COLUMNAS DE LA ENCUESTA

column_dictionary = pd.DataFrame({
    "posicion": range(len(df_starwars_raw.columns)),
    "columna_original": df_starwars_raw.columns,
    "metadata_fila_0": metadata_row.values,
    "nulos_en_respuestas": df_survey.isna().sum().values,
    "porcentaje_nulos_en_respuestas": (df_survey.isna().mean() * 100).round(2).values
})

display(column_dictionary)

,posicion,columna_original,metadata_fila_0,nulos_en_respuestas,porcentaje_nulos_en_respuestas
0,0,respondentid,NaN,0,0.00
1,1,have you seen any of the 6 films in the star w...,response,0,0.00
2,2,do you consider yourself to be a fan of the st...,response,350,29.51
3,3,which of the following star wars films have yo...,star wars: episode i the phantom menace,513,43.25
4,4,Unnamed: 4,star wars: episode ii attack of the clones,615,51.85
5,5,Unnamed: 5,star wars: episode iii revenge of the sith,636,53.63
6,6,Unnamed: 6,star wars: episode iv a new hope,579,48.82
7,7,Unnamed: 7,star wars: episode v the empire strikes back,428,36.09
8,8,Unnamed: 8,star wars: episode vi return of the jedi,448,37.77
9,9,please rank the star wars films in order of pr...,star wars: episode i the phantom menace,351,29.60


In [8]:

# 5. REVISIÓN DE COLUMNAS UNNAMED


unnamed_columns = [
    col for col in df_starwars_raw.columns 
    if "unnamed" in str(col).lower()
]

print("Número de columnas Unnamed:", len(unnamed_columns))

unnamed_dictionary = column_dictionary[
    column_dictionary["columna_original"].isin(unnamed_columns)
]

display(unnamed_dictionary)

Número de columnas Unnamed: 23


,posicion,columna_original,metadata_fila_0,nulos_en_respuestas,porcentaje_nulos_en_respuestas
4,4,Unnamed: 4,star wars: episode ii attack of the clones,615,51.85
5,5,Unnamed: 5,star wars: episode iii revenge of the sith,636,53.63
6,6,Unnamed: 6,star wars: episode iv a new hope,579,48.82
7,7,Unnamed: 7,star wars: episode v the empire strikes back,428,36.09
8,8,Unnamed: 8,star wars: episode vi return of the jedi,448,37.77
10,10,Unnamed: 10,star wars: episode ii attack of the clones,350,29.51
11,11,Unnamed: 11,star wars: episode iii revenge of the sith,351,29.60
12,12,Unnamed: 12,star wars: episode iv a new hope,350,29.51
13,13,Unnamed: 13,star wars: episode v the empire strikes back,350,29.51
14,14,Unnamed: 14,star wars: episode vi return of the jedi,350,29.51


In [9]:

# 6. NORMALIZAR NOMBRES DE COLUMNAS DE LA ENCUESTA


df_survey = normalize_column_names(df_survey)

print("Columnas normalizadas:")
for i, col in enumerate(df_survey.columns):
    print(i, "->", col)

Columnas normalizadas:
0 -> respondentid
1 -> have_you_seen_any_of_the_6_films_in_the_star_wars_franchise
2 -> do_you_consider_yourself_to_be_a_fan_of_the_star_wars_film_franchise
3 -> which_of_the_following_star_wars_films_have_you_seen_please_select_all_that_apply
4 -> unnamed_4
5 -> unnamed_5
6 -> unnamed_6
7 -> unnamed_7
8 -> unnamed_8
9 -> please_rank_the_star_wars_films_in_order_of_preference_with_1_being_your_favorite_film_in_the_franchise_and_6_being_your_least_favorite_film
10 -> unnamed_10
11 -> unnamed_11
12 -> unnamed_12
13 -> unnamed_13
14 -> unnamed_14
15 -> please_state_whether_you_view_the_following_characters_favorably
16 -> unfavorably
17 -> or_are_unfamiliar_with_him_her
18 -> unnamed_18
19 -> unnamed_19
20 -> unnamed_20
21 -> unnamed_21
22 -> unnamed_22
23 -> unnamed_23
24 -> unnamed_24
25 -> unnamed_25
26 -> unnamed_26
27 -> unnamed_27
28 -> unnamed_28
29 -> unnamed_29
30 -> unnamed_30
31 -> which_character_shot_first
32 -> are_you_familiar_with_the_expanded_univer

In [10]:

# 7. LIMPIEZA BÁSICA DE TEXTOS
'''quita espacios al principio/final
convierte textos vacíos en NaN
convierte "nan", "none", "null" en NaN real
'''

df_survey = clean_text_columns(df_survey)

display(df_survey.head())

,respondentid,have_you_seen_any_of_the_6_films_in_the_star_wars_franchise,do_you_consider_yourself_to_be_a_fan_of_the_star_wars_film_franchise,which_of_the_following_star_wars_films_have_you_seen_please_select_all_that_apply,unnamed_4,unnamed_5,unnamed_6,unnamed_7,unnamed_8,please_rank_the_star_wars_films_in_order_of_preference_with_1_being_your_favorite_film_in_the_franchise_and_6_being_your_least_favorite_film,unnamed_10,unnamed_11,unnamed_12,unnamed_13,unnamed_14,please_state_whether_you_view_the_following_characters_favorably,unfavorably,or_are_unfamiliar_with_him_her,unnamed_18,unnamed_19,unnamed_20,unnamed_21,unnamed_22,unnamed_23,unnamed_24,unnamed_25,unnamed_26,unnamed_27,unnamed_28,unnamed_29,unnamed_30,which_character_shot_first,are_you_familiar_with_the_expanded_universe,do_you_consider_yourself_to_be_a_fan_of_the_expanded_universe,do_you_consider_yourself_to_be_a_fan_of_the_star_trek_franchise,gender,age,household_income,education,location_census_region
0,3.292880e+09,yes,yes,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,star wars: episode iv a new hope,star wars: episode v the empire strikes back,star wars: episode vi return of the jedi,3,2,1,4,5,6,very favorably,very favorably,very favorably,very favorably,very favorably,very favorably,very favorably,unfamiliar (n/a),unfamiliar (n/a),very favorably,very favorably,very favorably,very favorably,very favorably,i don't understand this question,yes,no,no,male,18-29,NaN,high school degree,south atlantic,NaN,NaN
1,3.292880e+09,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yes,male,18-29,$0 - $24,999,bachelor degree,west south central,NaN
2,3.292765e+09,yes,no,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,NaN,NaN,NaN,1,2,3,4,5,6,somewhat favorably,somewhat favorably,somewhat favorably,somewhat favorably,somewhat favorably,unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),unfamiliar (n/a),i don't understand this question,no,NaN,no,male,18-29,$0 - $24,999,high school degree,west north central,NaN
3,3.292763e+09,yes,yes,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,star wars: episode iv a new hope,star wars: episode v the empire strikes back,star wars: episode vi return of the jedi,5,6,1,2,4,3,very favorably,very favorably,very favorably,very favorably,very favorably,somewhat favorably,very favorably,somewhat favorably,somewhat unfavorably,very favorably,very favorably,very favorably,very favorably,very favorably,i don't understand this question,no,NaN,yes,male,18-29,$100,000 - $149,999,some college or associate degree,west north central
4,3.292731e+09,yes,yes,star wars: episode i the phantom menace,star wars: episode ii attack of the clones,star wars: episode iii revenge of the sith,star wars: episode iv a new hope,star wars: episode v the empire strikes back,star wars: episode vi return of the jedi,5,4,6,2,1,3,very favorably,somewhat favorably,somewhat favorably,somewhat unfavorably,very favorably,very unfavorably,somewhat favorably,neither favorably nor unfavorably (neutral),very favorably,somewhat favorably,somewhat favorably,very unfavorably,somewhat favorably,somewhat favorably,greedo,yes,no,no,male,18-29,$100,000 - $149,999,some college or associate degree,west north central


In [11]:

# 9. REVISIÓN DE VALORES ÚNICOS POR COLUMNA

'''nombre de columna
número de valores distintos
primeros valores únicos'''

for col in df_survey.columns:
    print("\n" + "=" * 80)
    print("COLUMNA:", col)
    print("Nº valores únicos:", df_survey[col].nunique(dropna=True))
    print(df_survey[col].dropna().unique()[:10])


COLUMNA: respondentid
Nº valores únicos: 1186
[3.29288000e+09 3.29287954e+09 3.29276527e+09 3.29276312e+09
 3.29273122e+09 3.29271938e+09 3.29268479e+09 3.29266373e+09
 3.29265404e+09 3.29264042e+09]

COLUMNA: have_you_seen_any_of_the_6_films_in_the_star_wars_franchise
Nº valores únicos: 2
<StringArray>
['yes', 'no']
Length: 2, dtype: str

COLUMNA: do_you_consider_yourself_to_be_a_fan_of_the_star_wars_film_franchise
Nº valores únicos: 2
<StringArray>
['yes', 'no']
Length: 2, dtype: str

COLUMNA: which_of_the_following_star_wars_films_have_you_seen_please_select_all_that_apply
Nº valores únicos: 1
<StringArray>
['star wars: episode i  the phantom menace']
Length: 1, dtype: str

COLUMNA: unnamed_4
Nº valores únicos: 1
<StringArray>
['star wars: episode ii  attack of the clones']
Length: 1, dtype: str

COLUMNA: unnamed_5
Nº valores únicos: 1
<StringArray>
['star wars: episode iii  revenge of the sith']
Length: 1, dtype: str

COLUMNA: unnamed_6
Nº valores únicos: 1
<StringArray>
['star wa

In [12]:

# 10. IDENTIFICAR BLOQUES DE COLUMNAS POR POSICIÓN

#Con esta salida vamos a decidir qué columnas pertenecen a cada bloque

for i, col in enumerate(df_survey.columns):
    print(i, "->", col)

0 -> respondentid
1 -> have_you_seen_any_of_the_6_films_in_the_star_wars_franchise
2 -> do_you_consider_yourself_to_be_a_fan_of_the_star_wars_film_franchise
3 -> which_of_the_following_star_wars_films_have_you_seen_please_select_all_that_apply
4 -> unnamed_4
5 -> unnamed_5
6 -> unnamed_6
7 -> unnamed_7
8 -> unnamed_8
9 -> please_rank_the_star_wars_films_in_order_of_preference_with_1_being_your_favorite_film_in_the_franchise_and_6_being_your_least_favorite_film
10 -> unnamed_10
11 -> unnamed_11
12 -> unnamed_12
13 -> unnamed_13
14 -> unnamed_14
15 -> please_state_whether_you_view_the_following_characters_favorably
16 -> unfavorably
17 -> or_are_unfamiliar_with_him_her
18 -> unnamed_18
19 -> unnamed_19
20 -> unnamed_20
21 -> unnamed_21
22 -> unnamed_22
23 -> unnamed_23
24 -> unnamed_24
25 -> unnamed_25
26 -> unnamed_26
27 -> unnamed_27
28 -> unnamed_28
29 -> unnamed_29
30 -> unnamed_30
31 -> which_character_shot_first
32 -> are_you_familiar_with_the_expanded_universe
33 -> do_you_conside

### Revisión de columnas desplazadas en la encuesta

Después de realizar el análisis estructural anterior, detectamos que a partir de la columna 29 existe un desplazamiento en los encabezados de la encuesta. Los nombres de las columnas no coinciden con el contenido real que aparece en sus valores.

Concretamente, observamos que las respuestas de la columna 29 corresponden realmente a la pregunta que aparece nombrada en la columna 31. Es decir, desde ese punto los encabezados parecen estar desplazados dos posiciones hacia la derecha:

- La pregunta de la columna 31 corresponde realmente a los valores de la columna 29.
- La pregunta de la columna 32 corresponde realmente a los valores de la columna 30.
- La pregunta de la columna 33 corresponde realmente a los valores de la columna 31.
- Y así sucesivamente en las columnas finales del dataset.

Por este motivo, decidimos no fiarnos únicamente del nombre original de las columnas, sino analizar también sus valores únicos para identificar qué representa realmente cada una.

Un ejemplo claro es la columna 29, que aparece como `unnamed_29`, pero contiene respuestas como `han`, `greedo` o `i don't understand this question`. Por tanto, esta columna se interpreta como la pregunta `which_character_shot_first`.

El desplazamiento afecta especialmente a las preguntas finales y a las variables demográficas. Sin embargo, en las dos últimas columnas del dataset encontramos una dificultad adicional: ambas contienen respuestas relacionadas con lugares geográficos o regiones, por lo que no queda completamente claro a qué variable original corresponde cada una.

### Decisión sobre las dos últimas columnas

Como las dos últimas columnas contienen valores geográficos, se decide tratarlas con precaución. En lugar de eliminarlas directamente, se conservarán temporalmente para analizarlas con más detalle.

La decisión inicial será:

1. Revisar sus valores únicos.
2. Compararlas con las categorías esperadas de `location_census_region`.
3. Comprobar si una de ellas contiene claramente regiones censales válidas.
4. Mantener como `location_census_region` la columna que tenga mayor coherencia geográfica.
5. Dejar la otra como columna auxiliar o descartarla si se confirma que es duplicada, residual o resultado del desplazamiento de encabezados.

Por tanto, estas columnas no se eliminarán en la primera limpieza. Se mantendrán como columnas pendientes de validación hasta confirmar si aportan información útil o si deben excluirse del análisis final.

In [13]:

# AGRUPACIÓN DE COLUMNAS DE LA ENCUESTA STAR WARS

# Preguntas generales principales.
# Incluyen preguntas de sí/no y preguntas generales de cultura Star Wars.
general_columns = [
    df_survey.columns[1],   # ha visto alguna película de Star Wars
    df_survey.columns[2],   # se considera fan de Star Wars
    df_survey.columns[29],  # quién disparó primero
    df_survey.columns[30],  # familiaridad con universo expandido
    df_survey.columns[31],  # fan del universo expandido
    df_survey.columns[32],  # fan de Star Trek
]

# Columnas de películas vistas.
# Estas columnas indican qué películas ha visto cada persona.
# Si hay valor, normalmente significa que la ha visto; si hay NaN, no la ha visto.
seen_movies_columns = list(df_survey.columns[3:9])

# Columnas de ranking de películas.
# Valores del 1 al 6, donde 1 suele significar favorita y 6 menos favorita.
ranking_columns = list(df_survey.columns[9:15])

# Columnas de opinión sobre personajes.
# Contienen valores tipo:
# very favorably, somewhat favorably, neutral, unfavorable, unfamiliar...
character_opinion_columns = list(df_survey.columns[15:29])

# Columnas demográficas.
# OJO: en este dataset algunas columnas vienen desplazadas,
# por eso después las renombraremos manualmente.
demographic_columns = list(df_survey.columns[33:40])


# Guardamos todos los grupos en un diccionario para poder revisarlos fácilmente.
survey_column_groups = {
    "general": general_columns,
    "seen_movies": seen_movies_columns,
    "ranking": ranking_columns,
    "character_opinion": character_opinion_columns,
    "demographic": demographic_columns,
}


# Mostramos los grupos para comprobar que están bien.
for group_name, columns in survey_column_groups.items():
    print("\n" + "=" * 70)
    print(group_name.upper())
    print("=" * 70)
    for col in columns:
        print(col)



GENERAL
have_you_seen_any_of_the_6_films_in_the_star_wars_franchise
do_you_consider_yourself_to_be_a_fan_of_the_star_wars_film_franchise
unnamed_29
unnamed_30
which_character_shot_first
are_you_familiar_with_the_expanded_universe

SEEN_MOVIES
which_of_the_following_star_wars_films_have_you_seen_please_select_all_that_apply
unnamed_4
unnamed_5
unnamed_6
unnamed_7
unnamed_8

RANKING
please_rank_the_star_wars_films_in_order_of_preference_with_1_being_your_favorite_film_in_the_franchise_and_6_being_your_least_favorite_film
unnamed_10
unnamed_11
unnamed_12
unnamed_13
unnamed_14

CHARACTER_OPINION
please_state_whether_you_view_the_following_characters_favorably
unfavorably
or_are_unfamiliar_with_him_her
unnamed_18
unnamed_19
unnamed_20
unnamed_21
unnamed_22
unnamed_23
unnamed_24
unnamed_25
unnamed_26
unnamed_27
unnamed_28

DEMOGRAPHIC
do_you_consider_yourself_to_be_a_fan_of_the_expanded_universe
do_you_consider_yourself_to_be_a_fan_of_the_star_trek_franchise
gender
age
household_income
educ

In [14]:

# RENOMBRADO MANUAL DE COLUMNAS DE LA ENCUESTA


survey_column_mapping = {
    # Identificador
    df_survey.columns[0]: "respondent_id",

    # Preguntas generales
    df_survey.columns[1]: "has_seen_any_star_wars_film",
    df_survey.columns[2]: "is_star_wars_fan",

    # Películas vistas
    df_survey.columns[3]: "seen_episode_i_the_phantom_menace",
    df_survey.columns[4]: "seen_episode_ii_attack_of_the_clones",
    df_survey.columns[5]: "seen_episode_iii_revenge_of_the_sith",
    df_survey.columns[6]: "seen_episode_iv_a_new_hope",
    df_survey.columns[7]: "seen_episode_v_the_empire_strikes_back",
    df_survey.columns[8]: "seen_episode_vi_return_of_the_jedi",

    # Ranking de películas
    df_survey.columns[9]: "rank_episode_i_the_phantom_menace",
    df_survey.columns[10]: "rank_episode_ii_attack_of_the_clones",
    df_survey.columns[11]: "rank_episode_iii_revenge_of_the_sith",
    df_survey.columns[12]: "rank_episode_iv_a_new_hope",
    df_survey.columns[13]: "rank_episode_v_the_empire_strikes_back",
    df_survey.columns[14]: "rank_episode_vi_return_of_the_jedi",

    # Opinión sobre personajes
    df_survey.columns[15]: "opinion_han_solo",
    df_survey.columns[16]: "opinion_luke_skywalker",
    df_survey.columns[17]: "opinion_princess_leia_organa",
    df_survey.columns[18]: "opinion_anakin_skywalker",
    df_survey.columns[19]: "opinion_obi_wan_kenobi",
    df_survey.columns[20]: "opinion_emperor_palpatine",
    df_survey.columns[21]: "opinion_darth_vader",
    df_survey.columns[22]: "opinion_lando_calrissian",
    df_survey.columns[23]: "opinion_boba_fett",
    df_survey.columns[24]: "opinion_c_3po",
    df_survey.columns[25]: "opinion_r2_d2",
    df_survey.columns[26]: "opinion_jar_jar_binks",
    df_survey.columns[27]: "opinion_padme_amidala",
    df_survey.columns[28]: "opinion_yoda",

    # Preguntas generales finales
    df_survey.columns[29]: "which_character_shot_first",
    df_survey.columns[30]: "is_familiar_with_expanded_universe",
    df_survey.columns[31]: "is_expanded_universe_fan",
    df_survey.columns[32]: "is_star_trek_fan",

    # Datos demográficos
    df_survey.columns[33]: "gender",
    df_survey.columns[34]: "age",
    df_survey.columns[35]: "household_income",
    df_survey.columns[36]: "education",
    df_survey.columns[37]: "location_census_region",
    df_survey.columns[38]: "extra_column_38",
    df_survey.columns[39]: "extra_column_39",
}

df_survey = df_survey.rename(columns=survey_column_mapping)

print("Columnas renombradas correctamente:")
for i, col in enumerate(df_survey.columns):
    print(i, "->", col)



Columnas renombradas correctamente:
0 -> respondent_id
1 -> has_seen_any_star_wars_film
2 -> is_star_wars_fan
3 -> seen_episode_i_the_phantom_menace
4 -> seen_episode_ii_attack_of_the_clones
5 -> seen_episode_iii_revenge_of_the_sith
6 -> seen_episode_iv_a_new_hope
7 -> seen_episode_v_the_empire_strikes_back
8 -> seen_episode_vi_return_of_the_jedi
9 -> rank_episode_i_the_phantom_menace
10 -> rank_episode_ii_attack_of_the_clones
11 -> rank_episode_iii_revenge_of_the_sith
12 -> rank_episode_iv_a_new_hope
13 -> rank_episode_v_the_empire_strikes_back
14 -> rank_episode_vi_return_of_the_jedi
15 -> opinion_han_solo
16 -> opinion_luke_skywalker
17 -> opinion_princess_leia_organa
18 -> opinion_anakin_skywalker
19 -> opinion_obi_wan_kenobi
20 -> opinion_emperor_palpatine
21 -> opinion_darth_vader
22 -> opinion_lando_calrissian
23 -> opinion_boba_fett
24 -> opinion_c_3po
25 -> opinion_r2_d2
26 -> opinion_jar_jar_binks
27 -> opinion_padme_amidala
28 -> opinion_yoda
29 -> which_character_shot_first

In [15]:

# GRUPOS DE COLUMNAS CON NOMBRES DEFINITIVOS


general_columns = [
    "has_seen_any_star_wars_film",
    "is_star_wars_fan",
    "which_character_shot_first",
    "is_familiar_with_expanded_universe",
    "is_expanded_universe_fan",
    "is_star_trek_fan",
]

seen_movies_columns = [
    "seen_episode_i_the_phantom_menace",
    "seen_episode_ii_attack_of_the_clones",
    "seen_episode_iii_revenge_of_the_sith",
    "seen_episode_iv_a_new_hope",
    "seen_episode_v_the_empire_strikes_back",
    "seen_episode_vi_return_of_the_jedi",
]

ranking_columns = [
    "rank_episode_i_the_phantom_menace",
    "rank_episode_ii_attack_of_the_clones",
    "rank_episode_iii_revenge_of_the_sith",
    "rank_episode_iv_a_new_hope",
    "rank_episode_v_the_empire_strikes_back",
    "rank_episode_vi_return_of_the_jedi",
]

character_opinion_columns = [
    "opinion_han_solo",
    "opinion_luke_skywalker",
    "opinion_princess_leia_organa",
    "opinion_anakin_skywalker",
    "opinion_obi_wan_kenobi",
    "opinion_emperor_palpatine",
    "opinion_darth_vader",
    "opinion_lando_calrissian",
    "opinion_boba_fett",
    "opinion_c_3po",
    "opinion_r2_d2",
    "opinion_jar_jar_binks",
    "opinion_padme_amidala",
    "opinion_yoda",
]

demographic_columns = [
    "gender",
    "age",
    "household_income",
    "education",
    "location_census_region",
]

survey_column_groups = {
    "general": general_columns,
    "seen_movies": seen_movies_columns,
    "ranking": ranking_columns,
    "character_opinion": character_opinion_columns,
    "demographic": demographic_columns,
}

for group_name, columns in survey_column_groups.items():
    print("\n" + "=" * 70)
    print(group_name.upper())
    print("=" * 70)
    for col in columns:
        print(col)


GENERAL
has_seen_any_star_wars_film
is_star_wars_fan
which_character_shot_first
is_familiar_with_expanded_universe
is_expanded_universe_fan
is_star_trek_fan

SEEN_MOVIES
seen_episode_i_the_phantom_menace
seen_episode_ii_attack_of_the_clones
seen_episode_iii_revenge_of_the_sith
seen_episode_iv_a_new_hope
seen_episode_v_the_empire_strikes_back
seen_episode_vi_return_of_the_jedi

RANKING
rank_episode_i_the_phantom_menace
rank_episode_ii_attack_of_the_clones
rank_episode_iii_revenge_of_the_sith
rank_episode_iv_a_new_hope
rank_episode_v_the_empire_strikes_back
rank_episode_vi_return_of_the_jedi

CHARACTER_OPINION
opinion_han_solo
opinion_luke_skywalker
opinion_princess_leia_organa
opinion_anakin_skywalker
opinion_obi_wan_kenobi
opinion_emperor_palpatine
opinion_darth_vader
opinion_lando_calrissian
opinion_boba_fett
opinion_c_3po
opinion_r2_d2
opinion_jar_jar_binks
opinion_padme_amidala
opinion_yoda

DEMOGRAPHIC
gender
age
household_income
education
location_census_region


In [16]:

# REVISIÓN DE VALORES ÚNICOS TRAS RENOMBRAR


for group_name, columns in survey_column_groups.items():
    print("\n" + "=" * 80)
    print(f"GRUPO: {group_name.upper()}")
    print("=" * 80)

    for col in columns:
        if col in df_survey.columns:
            print("\nCOLUMNA:", col)
            print("Nº valores únicos:", df_survey[col].nunique(dropna=True))
            print(df_survey[col].dropna().unique()[:10])


GRUPO: GENERAL

COLUMNA: has_seen_any_star_wars_film
Nº valores únicos: 2
<StringArray>
['yes', 'no']
Length: 2, dtype: str

COLUMNA: is_star_wars_fan
Nº valores únicos: 2
<StringArray>
['yes', 'no']
Length: 2, dtype: str

COLUMNA: which_character_shot_first
Nº valores únicos: 3
<StringArray>
['i don't understand this question', 'greedo', 'han']
Length: 3, dtype: str

COLUMNA: is_familiar_with_expanded_universe
Nº valores únicos: 2
<StringArray>
['yes', 'no']
Length: 2, dtype: str

COLUMNA: is_expanded_universe_fan
Nº valores únicos: 2
<StringArray>
['no', 'yes']
Length: 2, dtype: str

COLUMNA: is_star_trek_fan
Nº valores únicos: 2
<StringArray>
['no', 'yes']
Length: 2, dtype: str

GRUPO: SEEN_MOVIES

COLUMNA: seen_episode_i_the_phantom_menace
Nº valores únicos: 1
<StringArray>
['star wars: episode i  the phantom menace']
Length: 1, dtype: str

COLUMNA: seen_episode_ii_attack_of_the_clones
Nº valores únicos: 1
<StringArray>
['star wars: episode ii  attack of the clones']
Length: 1, dt

## 3. Limpieza final de la encuesta

Este bloque convierte el diagnostico estructural en una limpieza reproducible para Power BI: demografia corregida, peliculas vistas en 1/0, rankings numericos, opiniones puntuadas y variables auxiliares de negocio.

In [17]:
# LIMPIEZA FINAL DE LA ENCUESTA

# Partimos de df_survey, que ya tiene la fila 0 eliminada y las columnas renombradas.
df_survey_clean = df_survey.copy()

# 1. Reconstruccion de demografia.
# Los ingresos venian partidos por comas: $50,000 - $99,999 se reparte en varias columnas.
VALID_GENDERS = {"male", "female"}
VALID_AGES = {"18-29", "30-44", "45-60", "> 60"}
VALID_EDUCATION = {
    "less than high school degree",
    "high school degree",
    "some college or associate degree",
    "bachelor degree",
    "graduate degree",
}
VALID_REGIONS = {
    "new england", "middle atlantic", "east north central", "west north central",
    "south atlantic", "east south central", "west south central", "mountain", "pacific",
}


def _clean_token(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    return np.nan if value in ["", "nan", "none", "null"] else value


def _first_valid(tokens, valid_values):
    for token in tokens:
        if token in valid_values:
            return token
    return np.nan


def _rebuild_income(tokens):
    if "$0 - $24" in tokens and "999" in tokens:
        return "$0 - $24,999"
    if "$25" in tokens and "000 - $49" in tokens and "999" in tokens:
        return "$25,000 - $49,999"
    if "$50" in tokens and "000 - $99" in tokens and "999" in tokens:
        return "$50,000 - $99,999"
    if "$100" in tokens and "000 - $149" in tokens and "999" in tokens:
        return "$100,000 - $149,999"
    if "$150" in tokens and "000+" in tokens:
        return "$150,000+"
    return np.nan


def _rebuild_demographics(row):
    tail_cols = [
        "is_expanded_universe_fan", "is_star_trek_fan", "gender", "age",
        "household_income", "education", "location_census_region",
        "extra_column_38", "extra_column_39",
    ]
    tokens = [_clean_token(row[col]) for col in tail_cols if col in row.index]
    tokens = [token for token in tokens if isinstance(token, str)]
    return pd.Series({
        "gender": _first_valid(tokens, VALID_GENDERS),
        "age": _first_valid(tokens, VALID_AGES),
        "household_income": _rebuild_income(tokens),
        "education": _first_valid(tokens, VALID_EDUCATION),
        "location_census_region": _first_valid(tokens, VALID_REGIONS),
    })

fixed_demographics = df_survey_clean.apply(_rebuild_demographics, axis=1)
for col in fixed_demographics.columns:
    df_survey_clean[col] = fixed_demographics[col]

df_survey_clean = df_survey_clean.drop(
    columns=[col for col in ["extra_column_38", "extra_column_39"] if col in df_survey_clean.columns]
)

# 2. Peliculas vistas: texto = vista, NaN = no seleccionada.
for col in seen_movies_columns:
    df_survey_clean[col] = df_survey_clean[col].notna().astype(int)

df_survey_clean["total_movies_seen"] = df_survey_clean[seen_movies_columns].sum(axis=1)

# 3. Rankings de peliculas a numerico.
for col in ranking_columns:
    df_survey_clean[col] = pd.to_numeric(df_survey_clean[col], errors="coerce")

df_survey_clean["has_complete_movie_ranking"] = (
    df_survey_clean[ranking_columns].notna().sum(axis=1).eq(6).astype(int)
)

# 4. Opiniones de personajes a puntuacion numerica.
opinion_score_map = {
    "very favorably": 2,
    "somewhat favorably": 1,
    "neither favorably nor unfavorably (neutral)": 0,
    "somewhat unfavorably": -1,
    "very unfavorably": -2,
    "unfamiliar (n/a)": np.nan,
}
opinion_score_columns = []
for col in character_opinion_columns:
    score_col = col.replace("opinion_", "opinion_score_")
    df_survey_clean[score_col] = df_survey_clean[col].map(opinion_score_map)
    opinion_score_columns.append(score_col)

df_survey_clean["average_character_opinion_score"] = df_survey_clean[opinion_score_columns].mean(axis=1)

# 5. Variables auxiliares para analisis ejecutivo.
def yes_no_to_binary(value):
    if value == "yes":
        return 1
    if value == "no":
        return 0
    return np.nan

for col in [
    "has_seen_any_star_wars_film", "is_star_wars_fan",
    "is_familiar_with_expanded_universe", "is_expanded_universe_fan", "is_star_trek_fan",
]:
    df_survey_clean[col + "_binary"] = df_survey_clean[col].apply(yes_no_to_binary)


def movie_consumption_segment(total_movies):
    if total_movies == 0:
        return "no_movies_seen"
    if total_movies <= 2:
        return "low_consumption"
    if total_movies <= 4:
        return "medium_consumption"
    return "high_consumption"


df_survey_clean["movie_consumption_segment"] = df_survey_clean["total_movies_seen"].apply(movie_consumption_segment)
df_survey_clean["fan_segment"] = np.where(
    df_survey_clean["is_star_wars_fan"] == "yes", "fan",
    np.where(df_survey_clean["is_star_wars_fan"] == "no", "not_fan", "unknown")
)
df_survey_clean["has_demographic_info"] = df_survey_clean[[
    "gender", "age", "household_income", "education", "location_census_region"
]].notna().any(axis=1).astype(int)

print("Dimensiones encuesta limpia:", df_survey_clean.shape)
print("Duplicados completos:", df_survey_clean.duplicated().sum())
print("IDs duplicados:", df_survey_clean["respondent_id"].duplicated().sum())

print("\nDistribucion de peliculas vistas:")
display(df_survey_clean["total_movies_seen"].value_counts().sort_index())

print("\nCategorias demograficas corregidas:")
for col in ["gender", "age", "household_income", "education", "location_census_region"]:
    print("\n" + col)
    display(df_survey_clean[col].value_counts(dropna=False))

print("\nNulos principales tras limpieza:")
display(missing_report(df_survey_clean).head(20))

Dimensiones encuesta limpia: (1186, 63)
Duplicados completos: 0
IDs duplicados: 0

Distribucion de peliculas vistas:


total_movies_seen
0    351
1     56
2     85
3     99
4     72
5     52
6    471
Name: count, dtype: int64


Categorias demograficas corregidas:

gender


gender
female    549
male      497
NaN       140
Name: count, dtype: int64


age


...

education
some college or associate degree    328
bachelor degree                     321
graduate degree                     275
NaN                                 150
high school degree                  105
less than high school degree          7
Name: count, dtype: int64


location_census_region


location_census_region
east north central    181
pacific               175
south atlantic        170
NaN                   143
middle atlantic       122
west south central    110
west north central     93
mountain               79
new england            75
east south central     38
Name: count, dtype: int64


Nulos principales tras limpieza:


,nulos,porcentaje
is_expanded_universe_fan,973,82.04
is_expanded_universe_fan_binary,973,82.04
opinion_score_padme_amidala,536,45.19
opinion_score_emperor_palpatine,528,44.52
opinion_score_lando_calrissian,514,43.34
opinion_score_boba_fett,506,42.66
opinion_score_jar_jar_binks,474,39.97
opinion_score_anakin_skywalker,415,34.99
opinion_score_obi_wan_kenobi,378,31.87
opinion_score_c_3po,374,31.53


## 4. Tablas largas para Power BI

Para facilitar filtros, rankings y relaciones en Power BI, la encuesta se separa en una tabla de respondentes y tres tablas largas: peliculas vistas, ranking de peliculas y opiniones de personajes.

In [18]:
# TABLAS LARGAS PARA POWER BI

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

movie_labels = {
    "seen_episode_i_the_phantom_menace": "Episode I - The Phantom Menace",
    "seen_episode_ii_attack_of_the_clones": "Episode II - Attack of the Clones",
    "seen_episode_iii_revenge_of_the_sith": "Episode III - Revenge of the Sith",
    "seen_episode_iv_a_new_hope": "Episode IV - A New Hope",
    "seen_episode_v_the_empire_strikes_back": "Episode V - The Empire Strikes Back",
    "seen_episode_vi_return_of_the_jedi": "Episode VI - Return of the Jedi",
}
movie_keys = {
    "seen_episode_i_the_phantom_menace": "the_phantom_menace",
    "seen_episode_ii_attack_of_the_clones": "attack_of_the_clones",
    "seen_episode_iii_revenge_of_the_sith": "revenge_of_the_sith",
    "seen_episode_iv_a_new_hope": "a_new_hope",
    "seen_episode_v_the_empire_strikes_back": "the_empire_strikes_back",
    "seen_episode_vi_return_of_the_jedi": "return_of_the_jedi",
}
rank_labels = {col: movie_labels[col.replace("rank_", "seen_")] for col in ranking_columns}
rank_keys = {col: movie_keys[col.replace("rank_", "seen_")] for col in ranking_columns}
character_labels = {col: col.replace("opinion_", "").replace("_", " ").title() for col in character_opinion_columns}

survey_respondents = df_survey_clean[[
    "respondent_id", "has_seen_any_star_wars_film", "has_seen_any_star_wars_film_binary",
    "is_star_wars_fan", "is_star_wars_fan_binary", "fan_segment",
    "total_movies_seen", "movie_consumption_segment", "has_complete_movie_ranking",
    "which_character_shot_first", "is_familiar_with_expanded_universe",
    "is_familiar_with_expanded_universe_binary", "is_expanded_universe_fan",
    "is_expanded_universe_fan_binary", "is_star_trek_fan", "is_star_trek_fan_binary",
    "gender", "age", "household_income", "education", "location_census_region",
    "has_demographic_info", "average_character_opinion_score",
]].copy()

survey_movies_seen = df_survey_clean[["respondent_id"] + seen_movies_columns].melt(
    id_vars="respondent_id", var_name="movie_code", value_name="has_seen_movie"
)
survey_movies_seen["movie_title"] = survey_movies_seen["movie_code"].map(movie_labels)
survey_movies_seen["film_key"] = survey_movies_seen["movie_code"].map(movie_keys)
survey_movies_seen["episode_order"] = survey_movies_seen["movie_code"].map({col: i + 1 for i, col in enumerate(seen_movies_columns)})

survey_movie_rankings = df_survey_clean[["respondent_id"] + ranking_columns].melt(
    id_vars="respondent_id", var_name="movie_code", value_name="movie_rank"
)
survey_movie_rankings["movie_title"] = survey_movie_rankings["movie_code"].map(rank_labels)
survey_movie_rankings["film_key"] = survey_movie_rankings["movie_code"].map(rank_keys)
survey_movie_rankings["episode_order"] = survey_movie_rankings["movie_code"].map({col: i + 1 for i, col in enumerate(ranking_columns)})
survey_movie_rankings["has_ranking"] = survey_movie_rankings["movie_rank"].notna().astype(int)

survey_character_opinions = df_survey_clean[["respondent_id"] + character_opinion_columns].melt(
    id_vars="respondent_id", var_name="character_code", value_name="opinion_label"
)
survey_character_opinions["character_name"] = survey_character_opinions["character_code"].map(character_labels)
survey_character_opinions["opinion_score"] = survey_character_opinions["opinion_label"].map(opinion_score_map)
survey_character_opinions["has_character_opinion"] = survey_character_opinions["opinion_label"].notna().astype(int)
survey_character_opinions["is_unfamiliar"] = survey_character_opinions["opinion_label"].eq("unfamiliar (n/a)").astype(int)

print("survey_respondents:", survey_respondents.shape)
print("survey_movies_seen:", survey_movies_seen.shape)
print("survey_movie_rankings:", survey_movie_rankings.shape)
print("survey_character_opinions:", survey_character_opinions.shape)

display(survey_respondents.head())
display(survey_movies_seen.head())
display(survey_movie_rankings.head())
display(survey_character_opinions.head())

# Exportacion provisional para Power BI.
df_survey_clean.to_csv(PROCESSED_DIR / "survey_clean_wide.csv", index=False)
survey_respondents.to_csv(PROCESSED_DIR / "survey_respondents.csv", index=False)
survey_movies_seen.to_csv(PROCESSED_DIR / "survey_movies_seen.csv", index=False)
survey_movie_rankings.to_csv(PROCESSED_DIR / "survey_movie_rankings.csv", index=False)
survey_character_opinions.to_csv(PROCESSED_DIR / "survey_character_opinions.csv", index=False)

print("CSV de encuesta exportados en:", PROCESSED_DIR)

survey_respondents: (1186, 23)
survey_movies_seen: (7116, 6)
survey_movie_rankings: (7116, 7)
survey_character_opinions: (16604, 7)


,respondent_id,has_seen_any_star_wars_film,has_seen_any_star_wars_film_binary,is_star_wars_fan,is_star_wars_fan_binary,fan_segment,total_movies_seen,movie_consumption_segment,has_complete_movie_ranking,which_character_shot_first,is_familiar_with_expanded_universe,is_familiar_with_expanded_universe_binary,is_expanded_universe_fan,is_expanded_universe_fan_binary,is_star_trek_fan,is_star_trek_fan_binary,gender,age,household_income,education,location_census_region,has_demographic_info,average_character_opinion_score
0,3.292880e+09,yes,1,yes,1.0,fan,6,high_consumption,1,i don't understand this question,yes,1.0,no,0.0,no,0.0,male,18-29,NaN,high school degree,south atlantic,1,2.000000
1,3.292880e+09,no,0,NaN,NaN,unknown,0,no_movies_seen,0,NaN,NaN,NaN,NaN,NaN,yes,1.0,male,18-29,"$0 - $24,999",bachelor degree,west south central,1,NaN
2,3.292765e+09,yes,1,no,0.0,not_fan,3,medium_consumption,1,i don't understand this question,no,0.0,NaN,NaN,no,0.0,male,18-29,"$0 - $24,999",high school degree,west north central,1,1.000000
3,3.292763e+09,yes,1,yes,1.0,fan,6,high_consumption,1,i don't understand this question,no,0.0,NaN,NaN,yes,1.0,male,18-29,"$100,000 - $149,999",some college or associate degree,west north central,1,1.642857
4,3.292731e+09,yes,1,yes,1.0,fan,6,high_consumption,1,greedo,yes,1.0,no,0.0,no,0.0,male,18-29,"$100,000 - $149,999",some college or associate degree,west north central,1,0.571429


,respondent_id,movie_code,has_seen_movie,movie_title,film_key,episode_order
0,3.292880e+09,seen_episode_i_the_phantom_menace,1,Episode I - The Phantom Menace,the_phantom_menace,1
1,3.292880e+09,seen_episode_i_the_phantom_menace,0,Episode I - The Phantom Menace,the_phantom_menace,1
2,3.292765e+09,seen_episode_i_the_phantom_menace,1,Episode I - The Phantom Menace,the_phantom_menace,1
3,3.292763e+09,seen_episode_i_the_phantom_menace,1,Episode I - The Phantom Menace,the_phantom_menace,1
4,3.292731e+09,seen_episode_i_the_phantom_menace,1,Episode I - The Phantom Menace,the_phantom_menace,1


,respondent_id,movie_code,movie_rank,movie_title,film_key,episode_order,has_ranking
0,3.292880e+09,rank_episode_i_the_phantom_menace,3.0,Episode I - The Phantom Menace,the_phantom_menace,1,1
1,3.292880e+09,rank_episode_i_the_phantom_menace,NaN,Episode I - The Phantom Menace,the_phantom_menace,1,0
2,3.292765e+09,rank_episode_i_the_phantom_menace,1.0,Episode I - The Phantom Menace,the_phantom_menace,1,1
3,3.292763e+09,rank_episode_i_the_phantom_menace,5.0,Episode I - The Phantom Menace,the_phantom_menace,1,1
4,3.292731e+09,rank_episode_i_the_phantom_menace,5.0,Episode I - The Phantom Menace,the_phantom_menace,1,1


,respondent_id,character_code,opinion_label,character_name,opinion_score,has_character_opinion,is_unfamiliar
0,3.292880e+09,opinion_han_solo,very favorably,Han Solo,2.0,1,0
1,3.292880e+09,opinion_han_solo,NaN,Han Solo,NaN,0,0
2,3.292765e+09,opinion_han_solo,somewhat favorably,Han Solo,1.0,1,0
3,3.292763e+09,opinion_han_solo,very favorably,Han Solo,2.0,1,0
4,3.292731e+09,opinion_han_solo,very favorably,Han Solo,2.0,1,0


CSV de encuesta exportados en: data\processed


## 5. Revision general inicial de datasets del universo


In [19]:
df_characters.info()
df_characters.head()


<class 'pandas.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           112 non-null    int64  
 1   name         112 non-null    str    
 2   species      112 non-null    str    
 3   gender       108 non-null    str    
 4   height       83 non-null     float64
 5   weight       77 non-null     float64
 6   hair_color   76 non-null     str    
 7   eye_color    104 non-null    str    
 8   skin_color   110 non-null    str    
 9   year_born    80 non-null     str    
 10  homeworld    99 non-null     str    
 11  year_died    50 non-null     str    
 12  description  112 non-null    str    
dtypes: float64(2), int64(1), str(10)
memory usage: 11.5 KB


,id,name,species,gender,height,weight,hair_color,eye_color,skin_color,year_born,homeworld,year_died,description
0,1,Luke Skywalker,Human,Male,1.72,77.0,Blond,Blue,Light,19,Tatooine,34,The main protagonist of the original trilogy.
1,2,Leia Organa,Human,Female,1.50,49.0,Brown,Brown,Light,19,Alderaan,35,A leader in the Rebel Alliance and twin sister...
2,3,Darth Vader,Human,Male,2.02,136.0,NaN,Yellow,Pale,41,Tatooine,4,The Sith Lord formerly known as Anakin Skywalker.
3,4,Yoda,Yoda's species,Male,0.66,17.0,White,Brown,Green,896,Unknown,4,A wise and powerful Jedi Master.
4,5,Han Solo,Human,Male,1.80,80.0,Brown,Hazel,Light,29,Corellia,34,A smuggler turned hero in the Rebel Alliance.


In [20]:
df_planets.info()
df_planets.head()

<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               26 non-null     int64  
 1   name             26 non-null     str    
 2   diameter         11 non-null     float64
 3   rotation_period  12 non-null     float64
 4   orbital_period   12 non-null     float64
 5   gravity          26 non-null     str    
 6   population       11 non-null     float64
 7   climate          26 non-null     str    
 8   terrain          26 non-null     str    
 9   surface_water    12 non-null     float64
 10  residents        26 non-null     str    
 11  films            25 non-null     str    
dtypes: float64(5), int64(1), str(6)
memory usage: 2.6 KB


,id,name,diameter,rotation_period,orbital_period,gravity,population,climate,terrain,surface_water,residents,films
0,1,Tatooine,10465.0,23.0,304.0,1 standard,2.000000e+05,arid,desert,1.0,"Anakin Skywalker, Shmi Skywalker, Luke Skywalk...","A New Hope, The Phantom Menace, Attack of the ..."
1,2,Alderaan,12500.0,24.0,364.0,1 standard,2.000000e+09,temperate,"grasslands, mountains",40.0,Leia Organa,"A New Hope, Revenge of the Sith"
2,3,Naboo,12120.0,26.0,312.0,1 standard,4.500000e+09,temperate,"grassy hills, swamps, forests, mountains",12.0,"Padmé Amidala, Jar Jar Binks, Sheev Palpatine","The Phantom Menace, Attack of the Clones, Reve..."
3,4,Stewjon,NaN,NaN,NaN,1 standard,NaN,temperate,"grasslands, lakes",NaN,Obi-Wan Kenobi,NaN
4,5,Corellia,11000.0,25.0,329.0,1 standard,3.000000e+09,temperate,"plains, urban, forests, hills",70.0,Han Solo,Solo: A Star Wars Story


In [21]:
df_species.info()
df_species.head()


<class 'pandas.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                39 non-null     int64  
 1   name              39 non-null     str    
 2   classification    39 non-null     str    
 3   designation       39 non-null     str    
 4   average_height    38 non-null     float64
 5   skin_colors       39 non-null     str    
 6   hair_colors       11 non-null     str    
 7   eye_colors        39 non-null     str    
 8   average_lifespan  38 non-null     float64
 9   language          39 non-null     str    
 10  homeworld         38 non-null     str    
dtypes: float64(2), int64(1), str(8)
memory usage: 3.5 KB


,id,name,classification,designation,average_height,skin_colors,hair_colors,eye_colors,average_lifespan,language,homeworld
0,1,Human,Mammal,Sentient,1.80,"Light, Dark",Various,Various,79.0,Galactic Basic,Various
1,2,Yoda's species,Unknown,Sentient,0.66,Green,White,Brown,900.0,Galactic Basic,Unknown
2,3,Wookiee,Mammal,Sentient,2.28,Brown,Brown,Blue,400.0,Shyriiwook,Kashyyyk
3,4,Gungan,Amphibian,Sentient,1.96,Orange,NaN,Orange,70.0,Gungan,Naboo
4,5,Twi'lek,Mammal,Sentient,1.80,"Blue, Green, Red, Yellow",NaN,Various,80.0,Twi'leki,Ryloth


In [22]:
df_starships.info()
df_starships.head()

<class 'pandas.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      56 non-null     int64  
 1   name                    56 non-null     str    
 2   model                   56 non-null     str    
 3   manufacturer            53 non-null     str    
 4   cost_in_credits         16 non-null     float64
 5   length                  56 non-null     float64
 6   max_atmosphering_speed  52 non-null     float64
 7   crew                    52 non-null     float64
 8   passengers              52 non-null     float64
 9   cargo_capacity          48 non-null     float64
 10  consumables             48 non-null     str    
 11  hyperdrive_rating       51 non-null     float64
 12  MGLT                    48 non-null     float64
 13  starship_class          56 non-null     str    
 14  pilots                  37 non-null     str    
 15  fi

,id,name,model,manufacturer,cost_in_credits,length,max_atmosphering_speed,crew,passengers,cargo_capacity,consumables,hyperdrive_rating,MGLT,starship_class,pilots,films
0,1,Millennium Falcon,YT-1300 light freighter,Corellian Engineering Corporation,100000.0,34.75,1050.0,4.0,6.0,100000.0,2 months,0.5,75.0,Light freighter,"Han Solo, Chewbacca","A New Hope, The Empire Strikes Back, Return of..."
1,2,X-wing,T-65 X-wing starfighter,Incom Corporation,149999.0,12.50,1050.0,1.0,0.0,110.0,1 week,1.0,100.0,Starfighter,Luke Skywalker,"A New Hope, The Empire Strikes Back, Return of..."
2,3,TIE Fighter,Twin Ion Engine/Ln Starfighter,Sienar Fleet Systems,75000.0,8.99,1200.0,1.0,0.0,65.0,2 days,1.0,100.0,Starfighter,NaN,"A New Hope, The Empire Strikes Back, Return of..."
3,4,Star Destroyer,Imperial I-class Star Destroyer,Kuat Drive Yards,150000000.0,1600.00,975.0,47060.0,0.0,36000000.0,2 years,2.0,60.0,Capital ship,NaN,"A New Hope, The Empire Strikes Back, Return of..."
4,5,Slave 1,Firespray-31-class patrol and attack craft,Kuat Systems Engineering,NaN,21.50,1000.0,1.0,6.0,80000.0,1 month,3.0,70.0,Patrol craft,Boba Fett,"The Empire Strikes Back, Attack of the Clones,..."


In [23]:
df_vehicles.info()
df_vehicles.head()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      21 non-null     int64  
 1   name                    21 non-null     str    
 2   model                   21 non-null     str    
 3   manufacturer            21 non-null     str    
 4   cost_in_credits         21 non-null     float64
 5   length                  21 non-null     float64
 6   max_atmosphering_speed  21 non-null     float64
 7   crew                    21 non-null     int64  
 8   passengers              21 non-null     int64  
 9   cargo_capacity          21 non-null     float64
 10  consumables             16 non-null     str    
 11  vehicle_class           21 non-null     str    
 12  pilots                  12 non-null     str    
 13  films                   21 non-null     str    
dtypes: float64(4), int64(3), str(7)
memory usage: 2.4 KB


,id,name,model,manufacturer,cost_in_credits,length,max_atmosphering_speed,crew,passengers,cargo_capacity,consumables,vehicle_class,pilots,films
0,1,Snowspeeder,t-47 airspeeder,Incom corporation,0.0,4.5,650.0,2,0,10.0,none,airspeeder,"Luke Skywalker, Wedge Antilles",The Empire Strikes Back
1,2,TIE Fighter,Twin Ion Engine/Ln Starfighter,Sienar Fleet Systems,75000.0,6.3,1200.0,1,0,65.0,2 days,Starfighter,NaN,"A New Hope, The Empire Strikes Back, Return of..."
2,3,Sand Crawler,Digger Crawler,Corellia Mining Corporation,150000.0,36.8,30.0,46,30,50000.0,2 months,Wheeled,NaN,A New Hope
3,4,X-34 Landspeeder,X-34 Landspeeder,SoroSuub Corporation,10550.0,3.4,250.0,1,1,5.0,NaN,Landspeeder,Luke Skywalker,A New Hope
4,5,TIE Bomber,TIE/sa Bomber,Sienar Fleet Systems,86500.0,7.8,850.0,1,0,15.0,2 days,Bomber,NaN,The Empire Strikes Back


In [24]:
df_quotes.info()
df_quotes.head()

<class 'pandas.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   id              89 non-null     int64
 1   character_name  89 non-null     str  
 2   quote           89 non-null     str  
 3   source          89 non-null     str  
dtypes: int64(1), str(3)
memory usage: 2.9 KB


,id,character_name,quote,source
0,1,Luke Skywalker,May the Force be with you.,A New Hope
1,2,Darth Vader,I am your father.,The Empire Strikes Back
2,3,Yoda,"Do or do not, there is no try.",The Empire Strikes Back
3,4,Han Solo,I've got a bad feeling about this.,A New Hope
4,5,Admiral Ackbar,It's a trap!,Return of the Jedi


In [25]:
df_weapons.info()
df_weapons.head()

<class 'pandas.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               57 non-null     int64  
 1   name             57 non-null     str    
 2   model            57 non-null     str    
 3   manufacturer     57 non-null     str    
 4   cost_in_credits  32 non-null     float64
 5   length           52 non-null     float64
 6   type             57 non-null     str    
 7   description      57 non-null     str    
 8   films            57 non-null     str    
dtypes: float64(2), int64(1), str(6)
memory usage: 4.1 KB


,id,name,model,manufacturer,cost_in_credits,length,type,description,films
0,1,Lightsaber,Standard Lightsaber,Various,NaN,1.00,Melee,A weapon with a plasma blade powered by a kybe...,All Episodes
1,2,DL-44 Blaster Pistol,DL-44,BlasTech Industries,1500.0,0.30,Blaster,A powerful and highly modifiable blaster pistol.,"A New Hope, The Empire Strikes Back, Return of..."
2,3,E-11 Blaster Rifle,E-11,BlasTech Industries,1000.0,0.43,Blaster,Standard issue blaster rifle for Imperial stor...,"A New Hope, The Empire Strikes Back, Return of..."
3,4,Bowcaster,Wookiee Bowcaster,Various,NaN,1.50,Projectile,A traditional Wookiee weapon that fires explos...,"A New Hope, The Force Awakens, The Last Jedi"
4,5,Thermal Detonator,Model TD,Various,2000.0,0.10,Explosive,A highly destructive grenade.,"Return of the Jedi, The Force Awakens"


In [26]:
df_droids.info()
df_droids.head()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                14 non-null     int64  
 1   name              14 non-null     str    
 2   model             14 non-null     str    
 3   manufacturer      11 non-null     str    
 4   height            13 non-null     float64
 5   mass              11 non-null     str    
 6   sensor_color      14 non-null     str    
 7   plating_color     14 non-null     str    
 8   primary_function  14 non-null     str    
 9   films             13 non-null     str    
dtypes: float64(1), int64(1), str(8)
memory usage: 1.2 KB


,id,name,model,manufacturer,height,mass,sensor_color,plating_color,primary_function,films
0,1,R2-D2,R2 series,Industrial Automaton,0.96,32.0,Red,White/Blue,Astromech droid,"A New Hope, The Empire Strikes Back, Return of..."
1,2,C-3PO,3PO series,Cybot Galactica,1.71,75.0,Yellow,Gold,Protocol droid,"A New Hope, The Empire Strikes Back, Return of..."
2,3,BB-8,BB series,BB Astromech,0.67,18.0,Black,White/Orange,Astromech droid,"The Force Awakens, The Last Jedi, The Rise of ..."
3,4,K-2SO,KX series,Arakyd Industries,2.16,104.0,Red,Black,Security droid,Rogue One: A Star Wars Story
4,5,IG-88,IG series,Holowan Laboratories,2.00,140.0,Red,Silver,Assassin droid,The Empire Strikes Back


## 6. Limpieza y preparacion de datasets del universo

En esta fase se limpian los datasets internos del universo Star Wars para usarlos en Power BI.

La limpieza aplicada es comun y reproducible:

- Normalizacion de nombres de columnas.
- Limpieza de textos y falsos nulos (`unknown`, `none`, `n/a`, cadenas vacias).
- Conversion de columnas numericas.
- Conversion de fechas de peliculas.
- Revision de duplicados y nulos.
- Creacion de columnas auxiliares de presencia: numero de peliculas asociadas, numero de residentes, pilotos o apariciones cuando aplica.
- Exportacion de CSV limpios a `data/processed`.

In [27]:
# LIMPIEZA GENERAL DE DATASETS DEL UNIVERSO

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

UNKNOWN_VALUES = {
    "", "unknown", "n/a", "na", "none", "null", "nan", "not available", "not applicable"
}


def clean_universe_text_columns(df):
    """Limpia textos y convierte falsos nulos en NaN."""
    df = df.copy()
    text_columns = df.select_dtypes(include=["object", "string", "str"]).columns

    for col in text_columns:
        df[col] = df[col].apply(clean_text)
        df[col] = df[col].apply(
            lambda value: np.nan
            if isinstance(value, str) and value.strip().lower() in UNKNOWN_VALUES
            else value
        )

    return df


def to_numeric_columns(df, columns):
    """Convierte columnas a numerico si existen."""
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def count_list_items(value):
    """Cuenta elementos separados por coma en columnas tipo lista."""
    if pd.isna(value):
        return 0
    value = str(value).strip()
    if value == "":
        return 0
    if value.lower() == "all episodes":
        return 11
    return len([item for item in value.split(",") if item.strip()])


def add_list_count(df, source_col, target_col):
    """Crea una columna con numero de elementos en una lista textual."""
    df = df.copy()
    if source_col in df.columns:
        df[target_col] = df[source_col].apply(count_list_items)
    return df


def clean_year_column(value):
    """Extrae valores numericos de anos tipo '0 BBY' o '34'."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if not match:
        return np.nan
    return float(match.group())


FILM_TITLE_KEY_MAP = {
    "episode i: the phantom menace": "the_phantom_menace",
    "episode ii: attack of the clones": "attack_of_the_clones",
    "episode iii: revenge of the sith": "revenge_of_the_sith",
    "episode iv: a new hope": "a_new_hope",
    "episode v: the empire strikes back": "the_empire_strikes_back",
    "episode vi: return of the jedi": "return_of_the_jedi",
    "episode vii: the force awakens": "the_force_awakens",
    "episode viii: the last jedi": "the_last_jedi",
    "episode ix: the rise of skywalker": "the_rise_of_skywalker",
    "rogue one: a star wars story": "rogue_one",
    "solo: a star wars story": "solo",
}


def film_key_from_title(value):
    """Normaliza titulos de peliculas para cruzar universo, encuesta y negocio."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in FILM_TITLE_KEY_MAP:
        return FILM_TITLE_KEY_MAP[text]
    text = re.sub(r"^star wars[: ]+", "", text)
    text = re.sub(r"^ep\.?\s*(i|ii|iii|iv|v|vi|vii|viii|ix)[: ]+", "", text)
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


universe_raw_datasets = {
    "characters": df_characters,
    "films": df_films,
    "planets": df_planets,
    "species": df_species,
    "starships": df_starships,
    "vehicles": df_vehicles,
    "quotes": df_quotes,
    "weapons": df_weapons,
    "droids": df_droids,
}

universe_clean_datasets = {}

for dataset_name, dataset in universe_raw_datasets.items():
    df = normalize_column_names(dataset)
    df = clean_universe_text_columns(df)
    df = df.drop_duplicates().reset_index(drop=True)
    universe_clean_datasets[dataset_name] = df

# Conversiones especificas por dataset.

universe_clean_datasets["characters"] = to_numeric_columns(
    universe_clean_datasets["characters"],
    ["id", "height", "weight"],
)
for col in ["year_born", "year_died"]:
    if col in universe_clean_datasets["characters"].columns:
        universe_clean_datasets["characters"][col] = universe_clean_datasets["characters"][col].apply(clean_year_column)

universe_clean_datasets["films"]["release_date"] = pd.to_datetime(
    universe_clean_datasets["films"]["release_date"], errors="coerce"
)
universe_clean_datasets["films"]["release_year"] = universe_clean_datasets["films"]["release_date"].dt.year
universe_clean_datasets["films"]["film_key"] = universe_clean_datasets["films"]["title"].apply(film_key_from_title)

df_films_business_clean = normalize_column_names(df_films_business)
df_films_business_clean = clean_universe_text_columns(df_films_business_clean)
df_films_business_clean = df_films_business_clean.drop_duplicates(subset=["film_key"]).reset_index(drop=True)
df_films_business_clean["release_date"] = pd.to_datetime(df_films_business_clean["release_date"], errors="coerce")
df_films_business_clean["release_year"] = pd.to_numeric(df_films_business_clean["release_year"], errors="coerce").fillna(df_films_business_clean["release_date"].dt.year).astype("Int64")
df_films_business_clean = to_numeric_columns(
    df_films_business_clean,
    [
        "budget_usd", "domestic_box_office_usd", "worldwide_box_office_usd",
        "profit_estimated_usd", "roi",
    ],
)
df_films_business_clean["profit_estimated_usd"] = (
    df_films_business_clean["worldwide_box_office_usd"] - df_films_business_clean["budget_usd"]
)
df_films_business_clean["roi"] = (
    df_films_business_clean["profit_estimated_usd"] / df_films_business_clean["budget_usd"]
).round(4)

universe_clean_datasets["planets"] = to_numeric_columns(
    universe_clean_datasets["planets"],
    ["id", "diameter", "rotation_period", "orbital_period", "population", "surface_water"],
)
universe_clean_datasets["planets"] = add_list_count(universe_clean_datasets["planets"], "residents", "resident_count")
universe_clean_datasets["planets"] = add_list_count(universe_clean_datasets["planets"], "films", "film_count")

universe_clean_datasets["species"] = to_numeric_columns(
    universe_clean_datasets["species"],
    ["id", "average_height", "average_lifespan"],
)

universe_clean_datasets["starships"] = to_numeric_columns(
    universe_clean_datasets["starships"],
    [
        "id", "cost_in_credits", "length", "max_atmosphering_speed", "crew",
        "passengers", "cargo_capacity", "hyperdrive_rating", "mglt", "MGLT",
    ],
)
universe_clean_datasets["starships"] = add_list_count(universe_clean_datasets["starships"], "pilots", "pilot_count")
universe_clean_datasets["starships"] = add_list_count(universe_clean_datasets["starships"], "films", "film_count")

universe_clean_datasets["vehicles"] = to_numeric_columns(
    universe_clean_datasets["vehicles"],
    ["id", "cost_in_credits", "length", "max_atmosphering_speed", "crew", "passengers", "cargo_capacity"],
)
universe_clean_datasets["vehicles"] = add_list_count(universe_clean_datasets["vehicles"], "pilots", "pilot_count")
universe_clean_datasets["vehicles"] = add_list_count(universe_clean_datasets["vehicles"], "films", "film_count")

universe_clean_datasets["quotes"] = to_numeric_columns(universe_clean_datasets["quotes"], ["id"])

universe_clean_datasets["weapons"] = to_numeric_columns(
    universe_clean_datasets["weapons"],
    ["id", "cost_in_credits", "length"],
)
universe_clean_datasets["weapons"] = add_list_count(universe_clean_datasets["weapons"], "films", "film_count")

universe_clean_datasets["droids"] = to_numeric_columns(
    universe_clean_datasets["droids"],
    ["id", "height", "mass"],
)
universe_clean_datasets["droids"] = add_list_count(universe_clean_datasets["droids"], "films", "film_count")

# Recuperamos variables comodas para seguir trabajando en el notebook.
df_characters_clean = universe_clean_datasets["characters"]
df_films_clean = universe_clean_datasets["films"]
df_planets_clean = universe_clean_datasets["planets"]
df_species_clean = universe_clean_datasets["species"]
df_starships_clean = universe_clean_datasets["starships"]
df_vehicles_clean = universe_clean_datasets["vehicles"]
df_quotes_clean = universe_clean_datasets["quotes"]
df_weapons_clean = universe_clean_datasets["weapons"]
df_droids_clean = universe_clean_datasets["droids"]

print("Datasets del universo limpiados:")
for name, df in universe_clean_datasets.items():
    print(f"{name}: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"films_business: {df_films_business_clean.shape[0]} filas x {df_films_business_clean.shape[1]} columnas")

Datasets del universo limpiados:
characters: 112 filas x 13 columnas
films: 11 filas x 8 columnas
planets: 26 filas x 14 columnas
species: 39 filas x 11 columnas
starships: 56 filas x 18 columnas
vehicles: 21 filas x 16 columnas
quotes: 89 filas x 4 columnas
weapons: 57 filas x 10 columnas
droids: 14 filas x 11 columnas
films_business: 12 filas x 15 columnas


In [28]:
# REPORTE DE CALIDAD DE LOS DATASETS DEL UNIVERSO

universe_quality_summary = []
universe_missing_reports = {}

for name, df in universe_clean_datasets.items():
    duplicate_rows = df.duplicated().sum()
    duplicate_names = df["name"].duplicated().sum() if "name" in df.columns else np.nan
    total_cells = df.shape[0] * df.shape[1]
    total_missing = int(df.isna().sum().sum())
    missing_pct = round((total_missing / total_cells) * 100, 2) if total_cells else 0

    universe_quality_summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": duplicate_rows,
        "duplicate_names": duplicate_names,
        "total_missing": total_missing,
        "missing_pct": missing_pct,
    })

    universe_missing_reports[name] = missing_report(df)

universe_quality_summary = pd.DataFrame(universe_quality_summary)

display(universe_quality_summary)

print("Columnas con mas nulos por dataset:")
for name, report in universe_missing_reports.items():
    print("\n" + "=" * 70)
    print(name.upper())
    display(report.head(10))

,dataset,rows,columns,duplicate_rows,duplicate_names,total_missing,missing_pct
0,characters,112,13,0,0.0,231,15.87
1,films,11,8,0,NaN,0,0.00
2,planets,26,14,0,0.0,73,20.05
3,species,39,11,0,0.0,37,8.62
4,starships,56,18,0,0.0,103,10.22
5,vehicles,21,16,0,1.0,15,4.46
6,quotes,89,4,0,NaN,0,0.00
7,weapons,57,10,0,0.0,30,5.26
8,droids,14,11,0,0.0,9,5.84


Columnas con mas nulos por dataset:

CHARACTERS


,nulos,porcentaje
year_died,62,55.36
hair_color,36,32.14
weight,35,31.25
year_born,32,28.57
height,29,25.89
homeworld,19,16.96
eye_color,8,7.14
species,4,3.57
gender,4,3.57
skin_color,2,1.79



FILMS


,nulos,porcentaje


...

,nulos,porcentaje



WEAPONS


,nulos,porcentaje
cost_in_credits,25,43.86
length,5,8.77



DROIDS


,nulos,porcentaje
mass,4,28.57
manufacturer,3,21.43
height,1,7.14
films,1,7.14


In [29]:
# TABLA RESUMEN DE ACTIVOS DEL UNIVERSO PARA POWER BI

# Esta tabla unifica activos de distinto tipo para rankings y KPIs ejecutivos.
# No sustituye a las tablas detalladas; sirve para comparar presencia general.

asset_tables = []

asset_sources = {
    "character": (df_characters_clean, "name"),
    "planet": (df_planets_clean, "name"),
    "species": (df_species_clean, "name"),
    "starship": (df_starships_clean, "name"),
    "vehicle": (df_vehicles_clean, "name"),
    "weapon": (df_weapons_clean, "name"),
    "droid": (df_droids_clean, "name"),
}

for asset_type, (df, name_col) in asset_sources.items():
    temp = pd.DataFrame({
        "asset_name": df[name_col] if name_col in df.columns else pd.Series([np.nan] * len(df)),
    })
    temp["asset_type"] = asset_type
    temp["source_dataset"] = asset_type + "s"
    temp["film_count"] = df["film_count"] if "film_count" in df.columns else np.nan
    temp["known_numeric_fields"] = df.select_dtypes(include="number").notna().sum(axis=1).values
    temp["missing_fields"] = df.isna().sum(axis=1).values
    temp["total_fields"] = df.shape[1]
    temp["data_completeness_pct"] = ((temp["total_fields"] - temp["missing_fields"]) / temp["total_fields"] * 100).round(2)
    asset_tables.append(temp)

universe_assets = pd.concat(asset_tables, ignore_index=True)

# Indicador simple de presencia interna. Mas adelante se podra combinar con afinidad de audiencia.
universe_assets["internal_presence_score"] = (
    universe_assets["film_count"].fillna(0) + universe_assets["known_numeric_fields"].fillna(0)
)

display(universe_assets.head())
print("Activos por tipo:")
display(universe_assets["asset_type"].value_counts())

,asset_name,asset_type,source_dataset,film_count,known_numeric_fields,missing_fields,total_fields,data_completeness_pct,internal_presence_score
0,Luke Skywalker,character,characters,NaN,5,0,13,100.00,5.0
1,Leia Organa,character,characters,NaN,5,0,13,100.00,5.0
2,Darth Vader,character,characters,NaN,5,1,13,92.31,5.0
3,Yoda,character,characters,NaN,5,1,13,92.31,5.0
4,Han Solo,character,characters,NaN,5,0,13,100.00,5.0


Activos por tipo:


asset_type
character    112
weapon        57
starship      56
species       39
planet        26
vehicle       21
droid         14
Name: count, dtype: int64

In [30]:
# EXPORTACION DE DATASETS DEL UNIVERSO LIMPIOS

for name, df in universe_clean_datasets.items():
    df.to_csv(PROCESSED_DIR / f"universe_{name}_clean.csv", index=False)

universe_quality_summary.to_csv(PROCESSED_DIR / "universe_quality_summary.csv", index=False)
universe_assets.to_csv(PROCESSED_DIR / "universe_assets.csv", index=False)
df_films_business_clean.to_csv(PROCESSED_DIR / "films_business_clean.csv", index=False)

print("CSV del universo exportados en:", PROCESSED_DIR)
print("Archivos creados:")
for file in sorted(PROCESSED_DIR.glob("universe_*.csv")):
    print("-", file.name)
print("- films_business_clean.csv")

CSV del universo exportados en:

 data\processed
Archivos creados:


- universe_assets.csv
- universe_characters_clean.csv
- universe_droids_clean.csv
- universe_films_clean.csv
- universe_planets_clean.csv
- universe_quality_summary.csv
- universe_quotes_clean.csv
- universe_species_clean.csv
- universe_starships_clean.csv
- universe_vehicles_clean.csv
- universe_weapons_clean.csv
- films_business_clean.csv


## 7. EDA ejecutivo

Con los datos ya limpios, este bloque realiza el analisis exploratorio orientado al dashboard final.

El objetivo no es solo describir los datos, sino convertirlos en lectura de negocio:

- Que peliculas y personajes tienen mas traccion en audiencia.
- Que activos internos existen en el universo Star Wars.
- Donde hay problemas de calidad, nulos o sesgos.
- Que elementos tienen mas potencial para merchandising.

In [31]:
# EDA 1: AUDIENCIA Y PERCEPCION

survey_kpis = pd.DataFrame([{
    "respondents": len(survey_respondents),
    "seen_any_star_wars_pct": round(survey_respondents["has_seen_any_star_wars_film_binary"].mean() * 100, 2),
    "star_wars_fan_pct": round(survey_respondents["is_star_wars_fan_binary"].mean() * 100, 2),
    "avg_movies_seen": round(survey_respondents["total_movies_seen"].mean(), 2),
    "complete_movie_ranking_pct": round(survey_respondents["has_complete_movie_ranking"].mean() * 100, 2),
    "with_demographic_info_pct": round(survey_respondents["has_demographic_info"].mean() * 100, 2),
}])

movie_views_summary = (
    survey_movies_seen
    .groupby(["episode_order", "movie_title", "film_key"], as_index=False)
    .agg(
        viewers=("has_seen_movie", "sum"),
        respondents=("respondent_id", "nunique"),
    )
)
movie_views_summary["view_rate_pct"] = (
    movie_views_summary["viewers"] / movie_views_summary["respondents"] * 100
).round(2)

movie_rank_summary = (
    survey_movie_rankings
    .dropna(subset=["movie_rank"])
    .groupby(["episode_order", "movie_title", "film_key"], as_index=False)
    .agg(
        avg_rank=("movie_rank", "mean"),
        median_rank=("movie_rank", "median"),
        ranking_responses=("respondent_id", "nunique"),
        first_place_votes=("movie_rank", lambda s: (s == 1).sum()),
    )
)
movie_rank_summary["avg_rank"] = movie_rank_summary["avg_rank"].round(2)
movie_rank_summary["first_place_pct"] = (
    movie_rank_summary["first_place_votes"] / movie_rank_summary["ranking_responses"] * 100
).round(2)
movie_rank_summary["preference_score"] = (7 - movie_rank_summary["avg_rank"]).round(2)

character_opinion_summary = (
    survey_character_opinions
    .groupby("character_name", as_index=False)
    .agg(
        avg_opinion_score=("opinion_score", "mean"),
        opinion_responses=("opinion_label", lambda s: s.notna().sum()),
        favorable_responses=("opinion_score", lambda s: (s > 0).sum()),
        unfavorable_responses=("opinion_score", lambda s: (s < 0).sum()),
        unfamiliar_responses=("is_unfamiliar", "sum"),
        total_rows=("respondent_id", "count"),
    )
)
character_opinion_summary["avg_opinion_score"] = character_opinion_summary["avg_opinion_score"].round(3)
character_opinion_summary["favorable_pct"] = (
    character_opinion_summary["favorable_responses"] / character_opinion_summary["opinion_responses"] * 100
).round(2)
character_opinion_summary["unfavorable_pct"] = (
    character_opinion_summary["unfavorable_responses"] / character_opinion_summary["opinion_responses"] * 100
).round(2)
character_opinion_summary["unfamiliar_pct"] = (
    character_opinion_summary["unfamiliar_responses"] / character_opinion_summary["opinion_responses"] * 100
).round(2)
character_opinion_summary = character_opinion_summary.sort_values(
    ["avg_opinion_score", "favorable_pct"], ascending=False
)

fan_by_age = (
    survey_respondents
    .dropna(subset=["age"])
    .groupby("age", as_index=False)
    .agg(
        respondents=("respondent_id", "nunique"),
        fan_rate_pct=("is_star_wars_fan_binary", lambda s: round(s.mean() * 100, 2)),
        avg_movies_seen=("total_movies_seen", "mean"),
    )
)
fan_by_age["avg_movies_seen"] = fan_by_age["avg_movies_seen"].round(2)

fan_by_gender = (
    survey_respondents
    .dropna(subset=["gender"])
    .groupby("gender", as_index=False)
    .agg(
        respondents=("respondent_id", "nunique"),
        fan_rate_pct=("is_star_wars_fan_binary", lambda s: round(s.mean() * 100, 2)),
        avg_movies_seen=("total_movies_seen", "mean"),
    )
)
fan_by_gender["avg_movies_seen"] = fan_by_gender["avg_movies_seen"].round(2)

film_business_summary = df_films_business_clean.sort_values(
    ["worldwide_box_office_usd", "roi"], ascending=False
).reset_index(drop=True)

movie_commercial_audience_summary = (
    df_films_business_clean
    .merge(
        movie_views_summary[["film_key", "movie_title", "viewers", "respondents", "view_rate_pct"]],
        on="film_key", how="left"
    )
    .merge(
        movie_rank_summary[["film_key", "avg_rank", "preference_score", "ranking_responses", "first_place_pct"]],
        on="film_key", how="left"
    )
)
movie_commercial_audience_summary["is_in_survey"] = movie_commercial_audience_summary["movie_title"].notna().astype(int)

print("KPIs encuesta")
display(survey_kpis)

print("Peliculas mas vistas")
display(movie_views_summary.sort_values("view_rate_pct", ascending=False))

print("Ranking medio de peliculas: menor avg_rank = mejor preferencia")
display(movie_rank_summary.sort_values("avg_rank"))

print("Personajes mejor valorados")
display(character_opinion_summary.head(10))

print("Fan rate por edad")
display(fan_by_age)

print("Fan rate por genero")
display(fan_by_gender)

KPIs encuesta

,respondents,seen_any_star_wars_pct,star_wars_fan_pct,avg_movies_seen,complete_movie_ranking_pct,with_demographic_info_pct
0,1186,78.92,66.03,3.29,70.32,88.2


Peliculas mas vistas


,episode_order,movie_title,film_key,viewers,respondents,view_rate_pct
4,5,Episode V - The Empire Strikes Back,the_empire_strikes_back,758,1186,63.91
5,6,Episode VI - Return of the Jedi,return_of_the_jedi,738,1186,62.23
0,1,Episode I - The Phantom Menace,the_phantom_menace,673,1186,56.75
3,4,Episode IV - A New Hope,a_new_hope,607,1186,51.18
1,2,Episode II - Attack of the Clones,attack_of_the_clones,571,1186,48.15
2,3,Episode III - Revenge of the Sith,revenge_of_the_sith,550,1186,46.37


Ranking medio de peliculas: menor avg_rank = mejor preferencia


...

,character_name,avg_opinion_score,opinion_responses,favorable_responses,unfavorable_responses,unfamiliar_responses,total_rows,favorable_pct,unfavorable_pct,unfamiliar_pct
5,Han Solo,1.672,829,761,9,15,1186,91.80,1.09,1.81
9,Obi Wan Kenobi,1.632,825,750,15,17,1186,90.91,1.82,2.06
13,Yoda,1.630,826,749,16,10,1186,90.68,1.94,1.21
8,Luke Skywalker,1.581,831,771,16,6,1186,92.78,1.93,0.72
12,R2 D2,1.570,830,747,16,10,1186,90.00,1.93,1.20
11,Princess Leia Organa,1.555,831,757,18,8,1186,91.10,2.17,0.96
2,C 3Po,1.404,827,703,30,15,1186,85.01,3.63,1.81
0,Anakin Skywalker,0.776,823,514,122,52,1186,62.45,14.82,6.32
7,Lando Calrissian,0.637,820,365,71,148,1186,44.51,8.66,18.05
10,Padme Amidala,0.605,814,351,92,164,1186,43.12,11.30,20.15


Fan rate por edad


,age,respondents,fan_rate_pct,avg_movies_seen
0,18-29,218,68.89,4.24
1,30-44,268,72.46,3.94
2,45-60,291,64.17,3.66
3,> 60,269,58.55,2.90


Fan rate por genero


,gender,respondents,fan_rate_pct,avg_movies_seen
0,female,549,59.95,3.10
1,male,497,71.63,4.27


In [32]:
# EDA 2: CONTENIDO INTERNO DEL UNIVERSO STAR WARS

universe_overview = pd.DataFrame([
    {"asset_type": "characters", "count": len(df_characters_clean)},
    {"asset_type": "films", "count": len(df_films_clean)},
    {"asset_type": "planets", "count": len(df_planets_clean)},
    {"asset_type": "species", "count": len(df_species_clean)},
    {"asset_type": "starships", "count": len(df_starships_clean)},
    {"asset_type": "vehicles", "count": len(df_vehicles_clean)},
    {"asset_type": "weapons", "count": len(df_weapons_clean)},
    {"asset_type": "droids", "count": len(df_droids_clean)},
    {"asset_type": "quotes", "count": len(df_quotes_clean)},
])

character_species_summary = (
    df_characters_clean
    .groupby("species", dropna=False, as_index=False)
    .agg(characters=("id", "count"))
    .sort_values("characters", ascending=False)
)

character_gender_summary = (
    df_characters_clean
    .groupby("gender", dropna=False, as_index=False)
    .agg(characters=("id", "count"))
    .sort_values("characters", ascending=False)
)

character_homeworld_summary = (
    df_characters_clean
    .groupby("homeworld", dropna=False, as_index=False)
    .agg(characters=("id", "count"))
    .sort_values("characters", ascending=False)
)

planet_business_summary = df_planets_clean[[
    "name", "population", "diameter", "climate", "terrain", "resident_count", "film_count"
]].copy().sort_values(["film_count", "resident_count", "population"], ascending=False)

starship_business_summary = df_starships_clean[[
    "name", "starship_class", "manufacturer", "cost_in_credits", "length", "crew",
    "passengers", "cargo_capacity", "pilot_count", "film_count"
]].copy().sort_values(["film_count", "pilot_count", "cost_in_credits"], ascending=False)

weapon_business_summary = df_weapons_clean[[
    "name", "type", "manufacturer", "cost_in_credits", "length", "film_count"
]].copy().sort_values(["film_count", "cost_in_credits"], ascending=False)

quote_character_summary = (
    df_quotes_clean
    .groupby("character_name", as_index=False)
    .agg(quote_count=("quote", "count"))
    .sort_values("quote_count", ascending=False)
)

print("Resumen de activos internos")
display(universe_overview)

print("Top especies por numero de personajes")
display(character_species_summary.head(10))

print("Distribucion de genero de personajes")
display(character_gender_summary)

print("Top planetas por presencia narrativa")
display(planet_business_summary.head(10))

print("Top naves por presencia/capacidad comercial")
display(starship_business_summary.head(10))

print("Top armas por presencia")
display(weapon_business_summary.head(10))

print("Personajes con mas citas registradas")
display(quote_character_summary.head(10))

Resumen de activos internos


,asset_type,count
0,characters,112
1,films,11
2,planets,26
3,species,39
4,starships,56
5,vehicles,21
6,weapons,57
7,droids,14
8,quotes,89


Top especies por numero de personajes


,species,characters
10,Human,73
4,Dathomirian,4
5,Droid,4
26,NaN,4
21,Twi'lek,3
19,Togruta,2
14,Mon Calamari,2
2,Chiss,1
1,Besalisk,1
0,Ardennian,1


Distribucion de genero de personajes


...

,name,starship_class,manufacturer,cost_in_credits,length,crew,passengers,cargo_capacity,pilot_count,film_count
0,Millennium Falcon,Light freighter,Corellian Engineering Corporation,1.000000e+05,34.75,4.0,6.0,100000.0,2,6
1,X-wing,Starfighter,Incom Corporation,1.499990e+05,12.50,1.0,0.0,110.0,1,6
4,Slave 1,Patrol craft,Kuat Systems Engineering,NaN,21.50,1.0,6.0,80000.0,1,3
21,Slave II,Patrol craft,Kuat Systems Engineering,NaN,21.50,1.0,6.0,80000.0,1,3
3,Star Destroyer,Capital ship,Kuat Drive Yards,1.500000e+08,1600.00,47060.0,0.0,36000000.0,0,3
23,Imperial Star Destroyer,Capital ship,Kuat Drive Yards,1.500000e+08,1600.00,47060.0,0.0,36000000.0,0,3
7,Y-wing,Assault starfighter,Koensayr Manufacturing,1.350000e+05,23.40,2.0,0.0,110.0,0,3
2,TIE Fighter,Starfighter,Sienar Fleet Systems,7.500000e+04,8.99,1.0,0.0,65.0,0,3
52,N-1 Starfighter,Starfighter,Theed Palace Space Vessel Engineering Corps,2.000000e+05,11.00,1.0,1.0,NaN,2,2
8,Executor,Star Dreadnought,"Kuat Drive Yards, Fondor Shipyards",1.143350e+09,19000.00,279144.0,38000.0,250000000.0,1,2


Top armas por presencia


,name,type,manufacturer,cost_in_credits,length,film_count
0,Lightsaber,Melee,Various,NaN,1.00,11
1,DL-44 Blaster Pistol,Blaster,BlasTech Industries,1500.0,0.30,4
14,F-11D Blaster Rifle,Blaster,Sonn-Blas Corporation,2500.0,1.10,3
2,E-11 Blaster Rifle,Blaster,BlasTech Industries,1000.0,0.43,3
3,Bowcaster,Projectile,Various,NaN,1.50,3
37,Proton Torpedo,Projectile,Various,NaN,NaN,3
42,E-Web Heavy Blaster,Blaster,BlasTech Industries,5000.0,2.00,2
6,DC-15A Blaster Rifle,Blaster,BlasTech Industries,2500.0,1.26,2
4,Thermal Detonator,Explosive,Various,2000.0,0.10,2
28,DC-15S Blaster Carbine,Blaster,BlasTech Industries,1500.0,0.76,2


Personajes con mas citas registradas


,character_name,quote_count
9,Han Solo,17
13,Obi-Wan Kenobi,15
18,Yoda,11
6,Darth Vader,10
12,Luke Skywalker,9
11,Leia Organa,6
7,Emperor Palpatine,4
16,Qui-Gon Jinn,2
15,Padmé Amidala,2
2,Anakin Skywalker,2


In [33]:
# EDA 3: SESGOS, GOBERNANZA Y CALIDAD DEL DATO

if "universe_missing_reports" not in globals():
    if "universe_clean_datasets" not in globals():
        raise RuntimeError(
            "Ejecuta primero las celdas de limpieza/calidad de datasets del universo antes de esta celda."
        )
    universe_missing_reports = {
        dataset_name: missing_report(dataset)
        for dataset_name, dataset in universe_clean_datasets.items()
    }

survey_missing_top = missing_report(df_survey_clean).reset_index().rename(columns={"index": "column"})
survey_missing_top["dataset"] = "survey"

universe_missing_long = []
for dataset_name, report in universe_missing_reports.items():
    temp = report.reset_index().rename(columns={"index": "column"})
    temp["dataset"] = dataset_name
    universe_missing_long.append(temp)

universe_missing_long = pd.concat(universe_missing_long, ignore_index=True) if universe_missing_long else pd.DataFrame()

governance_missing_top = pd.concat(
    [survey_missing_top[["dataset", "column", "nulos", "porcentaje"]], universe_missing_long[["dataset", "column", "nulos", "porcentaje"]]],
    ignore_index=True,
).sort_values(["porcentaje", "nulos"], ascending=False)

survey_sample_bias = pd.DataFrame([
    {
        "risk": "Muestra orientada a personas que conocen Star Wars",
        "evidence": f"{survey_kpis.loc[0, 'seen_any_star_wars_pct']}% declara haber visto alguna pelicula.",
        "business_impact": "Puede sobreestimar la demanda real del publico general.",
    },
    {
        "risk": "Sesgo fan",
        "evidence": f"{survey_kpis.loc[0, 'star_wars_fan_pct']}% se declara fan entre respuestas validas.",
        "business_impact": "Las preferencias pueden favorecer personajes iconicos y no nichos de crecimiento.",
    },
    {
        "risk": "Datos demograficos incompletos",
        "evidence": f"{survey_kpis.loc[0, 'with_demographic_info_pct']}% tiene alguna informacion demografica util.",
        "business_impact": "La segmentacion por edad, genero, ingresos o region debe interpretarse con cautela.",
    },
    {
        "risk": "Datos internos incompletos",
        "evidence": "Planetas, naves, armas y droides tienen campos economicos o fisicos con nulos.",
        "business_impact": "El potencial comercial no debe calcularse solo con coste, tamano o capacidad.",
    },
])

print("Top problemas de nulos para pagina de gobernanza")
display(governance_missing_top.head(20))

print("Riesgos de sesgo detectados")
display(survey_sample_bias)

Top problemas de nulos para pagina de gobernanza

,dataset,column,nulos,porcentaje
0,survey,is_expanded_universe_fan,973,82.04
1,survey,is_expanded_universe_fan_binary,973,82.04
65,species,hair_colors,29,74.36
72,starships,cost_in_credits,40,71.43
59,planets,diameter,15,57.69
60,planets,population,15,57.69
49,characters,year_died,62,55.36
61,planets,rotation_period,14,53.85
62,planets,orbital_period,14,53.85
63,planets,surface_water,14,53.85


Riesgos de sesgo detectados


,risk,evidence,business_impact
0,Muestra orientada a personas que conocen Star ...,78.92% declara haber visto alguna pelicula.,Puede sobreestimar la demanda real del publico...
1,Sesgo fan,66.03% se declara fan entre respuestas validas.,Las preferencias pueden favorecer personajes i...
2,Datos demograficos incompletos,88.2% tiene alguna informacion demografica util.,"La segmentacion por edad, genero, ingresos o r..."
3,Datos internos incompletos,"Planetas, naves, armas y droides tienen campos...",El potencial comercial no debe calcularse solo...


In [34]:
# EDA 4: INDICE DE POTENCIAL DE MERCHANDISING

# Normalizamos nombres para cruzar opiniones de encuesta con activos internos.
def normalize_name_for_match(value):
    if pd.isna(value):
        return np.nan
    value = str(value).lower().strip()
    value = value.replace("princess leia organa", "leia organa")
    value = value.replace("c 3Po".lower(), "c-3po")
    value = value.replace("c 3po", "c-3po")
    value = value.replace("r2 d2", "r2-d2")
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")

character_opportunities = character_opinion_summary.copy()
character_opportunities["match_name"] = character_opportunities["character_name"].apply(normalize_name_for_match)

characters_for_merge = df_characters_clean.copy()
characters_for_merge["match_name"] = characters_for_merge["name"].apply(normalize_name_for_match)
characters_for_merge = characters_for_merge[[
    "match_name", "name", "species", "gender", "homeworld", "height", "weight"
]].rename(columns={"name": "universe_character_name"})

quote_for_merge = quote_character_summary.copy()
quote_for_merge["match_name"] = quote_for_merge["character_name"].apply(normalize_name_for_match)
quote_for_merge = quote_for_merge[["match_name", "quote_count"]]

asset_quality_for_merge = universe_assets[universe_assets["asset_type"] == "character"][[
    "asset_name", "data_completeness_pct", "internal_presence_score"
]].copy()
asset_quality_for_merge["match_name"] = asset_quality_for_merge["asset_name"].apply(normalize_name_for_match)
asset_quality_for_merge = asset_quality_for_merge.drop(columns=["asset_name"])

character_opportunities = character_opportunities.merge(characters_for_merge, on="match_name", how="left")
character_opportunities = character_opportunities.merge(quote_for_merge, on="match_name", how="left")
character_opportunities = character_opportunities.merge(asset_quality_for_merge, on="match_name", how="left")

character_opportunities["quote_count"] = character_opportunities["quote_count"].fillna(0)
character_opportunities["is_in_universe_dataset"] = character_opportunities["universe_character_name"].notna().astype(int)
character_opportunities["data_completeness_pct"] = character_opportunities["data_completeness_pct"].fillna(0)
character_opportunities["internal_presence_score"] = character_opportunities["internal_presence_score"].fillna(0) + character_opportunities["quote_count"]

# Escalado de componentes a 0-100.
character_opportunities["audience_affinity_score"] = (((character_opportunities["avg_opinion_score"] + 2) / 4) * 100).round(2)
character_opportunities["familiarity_score"] = (100 - character_opportunities["unfamiliar_pct"]).round(2)
max_presence = character_opportunities["internal_presence_score"].max()
character_opportunities["internal_presence_score_scaled"] = np.where(
    max_presence > 0,
    (character_opportunities["internal_presence_score"] / max_presence * 100).round(2),
    0,
)
character_opportunities["data_quality_score"] = character_opportunities["data_completeness_pct"].round(2)

character_opportunities["merchandising_potential_index"] = (
    character_opportunities["audience_affinity_score"] * 0.45
    + character_opportunities["familiarity_score"] * 0.25
    + character_opportunities["internal_presence_score_scaled"] * 0.20
    + character_opportunities["data_quality_score"] * 0.10
).round(2)

presence_median = character_opportunities["internal_presence_score_scaled"].median()
audience_median = character_opportunities["audience_affinity_score"].median()


def opportunity_quadrant(row):
    high_presence = row["internal_presence_score_scaled"] >= presence_median
    high_audience = row["audience_affinity_score"] >= audience_median
    if high_presence and high_audience:
        return "priority_campaign"
    if high_presence and not high_audience:
        return "repositioning_needed"
    if not high_presence and high_audience:
        return "hidden_opportunity"
    return "low_priority"


character_opportunities["opportunity_quadrant"] = character_opportunities.apply(opportunity_quadrant, axis=1)
character_opportunities = character_opportunities.sort_values("merchandising_potential_index", ascending=False)

movie_opportunities = movie_views_summary.merge(
    movie_rank_summary[["movie_title", "avg_rank", "preference_score", "first_place_pct"]],
    on="movie_title",
    how="left",
)
movie_opportunities["movie_campaign_score"] = (
    movie_opportunities["view_rate_pct"] * 0.45
    + (movie_opportunities["preference_score"] / 6 * 100) * 0.45
    + movie_opportunities["first_place_pct"] * 0.10
).round(2)
movie_opportunities = movie_opportunities.sort_values("movie_campaign_score", ascending=False)

print("Top personajes por indice de merchandising")
display(character_opportunities[[
    "character_name", "merchandising_potential_index", "opportunity_quadrant",
    "audience_affinity_score", "familiarity_score", "internal_presence_score_scaled",
    "data_quality_score", "avg_opinion_score", "quote_count", "species", "gender", "homeworld"
]].head(12))

print("Cuadrante de oportunidades")
display(character_opportunities["opportunity_quadrant"].value_counts())

print("Oportunidad por pelicula")
display(movie_opportunities)

Top personajes por indice de merchandising


,character_name,merchandising_potential_index,opportunity_quadrant,audience_affinity_score,familiarity_score,internal_presence_score_scaled,data_quality_score,avg_opinion_score,quote_count,species,gender,homeworld
0,Han Solo,95.86,priority_campaign,91.80,98.19,100.00,100.00,1.672,17.0,Human,Male,Corellia
1,Obi Wan Kenobi,93.53,priority_campaign,90.80,97.94,90.91,100.00,1.632,15.0,Human,Male,Stewjon
2,Yoda,89.31,priority_campaign,90.75,98.79,72.73,92.31,1.630,11.0,Yoda's species,Male,NaN
3,Luke Skywalker,87.83,priority_campaign,89.52,99.28,63.64,100.00,1.581,9.0,Human,Male,Tatooine
5,Princess Leia Organa,84.76,priority_campaign,88.88,99.04,50.00,100.00,1.555,6.0,Human,Female,Alderaan
10,Darth Vader,75.46,repositioning_needed,61.98,98.79,68.18,92.31,0.479,10.0,Human,Male,Tatooine
4,R2 D2,73.74,hidden_opportunity,89.25,98.80,13.64,61.54,1.570,0.0,Droid,NaN,Naboo
6,C 3Po,73.54,hidden_opportunity,85.10,98.19,22.73,61.54,1.404,2.0,Droid,NaN,Tatooine
7,Anakin Skywalker,71.01,low_priority,69.40,93.68,31.82,100.00,0.776,2.0,Human,Male,Tatooine
8,Lando Calrissian,64.84,low_priority,65.92,81.95,27.27,92.31,0.637,2.0,Human,Male,Socorro


Cuadrante de oportunidades


opportunity_quadrant
priority_campaign       5
low_priority            5
repositioning_needed    2
hidden_opportunity      2
Name: count, dtype: int64

Oportunidad por pelicula


,episode_order,movie_title,film_key,viewers,respondents,view_rate_pct,avg_rank,preference_score,first_place_pct,movie_campaign_score
4,5,Episode V - The Empire Strikes Back,the_empire_strikes_back,758,1186,63.91,2.51,4.49,34.57,65.89
5,6,Episode VI - Return of the Jedi,return_of_the_jedi,738,1186,62.23,3.05,3.95,17.46,59.37
3,4,Episode IV - A New Hope,a_new_hope,607,1186,51.18,3.27,3.73,24.40,53.45
0,1,Episode I - The Phantom Menace,the_phantom_menace,673,1186,56.75,3.73,3.27,15.45,51.61
1,2,Episode II - Attack of the Clones,attack_of_the_clones,571,1186,48.15,4.09,2.91,3.83,43.88
2,3,Episode III - Revenge of the Sith,revenge_of_the_sith,550,1186,46.37,4.34,2.66,4.31,41.25


In [35]:
# CONCLUSIONES AUTOMATICAS DEL EDA

best_movie_by_views = movie_views_summary.sort_values("view_rate_pct", ascending=False).iloc[0]
best_movie_by_rank = movie_rank_summary.sort_values("avg_rank").iloc[0]
best_character = character_opportunities.iloc[0]
best_business_movie = film_business_summary.iloc[0]
best_roi_movie = film_business_summary.sort_values("roi", ascending=False).iloc[0]
most_common_species = character_species_summary.iloc[0]
most_common_gender = character_gender_summary.iloc[0]

eda_conclusions = pd.DataFrame([
    {
        "area": "Audiencia",
        "finding": f"La pelicula mas vista es {best_movie_by_views['movie_title']} con un {best_movie_by_views['view_rate_pct']}% de visionado.",
        "business_reading": "Es una candidata fuerte para campanas de alto reconocimiento.",
    },
    {
        "area": "Preferencia",
        "finding": f"La pelicula con mejor ranking medio es {best_movie_by_rank['movie_title']} con avg_rank {best_movie_by_rank['avg_rank']}.",
        "business_reading": "La preferencia declarada no siempre coincide con la simple exposicion.",
    },
    {
        "area": "Personajes",
        "finding": f"El personaje con mayor indice de merchandising es {best_character['character_name']}.",
        "business_reading": "Conviene priorizar personajes con alta afinidad, familiaridad y presencia narrativa.",
    },
    {
        "area": "Representacion interna",
        "finding": f"La especie mas frecuente en characters es {most_common_species['species']} con {most_common_species['characters']} personajes.",
        "business_reading": "El universo esta concentrado en ciertos grupos, lo que puede limitar diversidad de campanas.",
    },
    {
        "area": "Negocio",
        "finding": f"La pelicula con mayor taquilla mundial es {best_business_movie['film_title']} con ${best_business_movie['worldwide_box_office_usd']:,.0f}.",
        "business_reading": "El rendimiento comercial complementa la lectura de audiencia y ayuda a priorizar oportunidades de franquicia.",
    },
    {
        "area": "Rentabilidad",
        "finding": f"La pelicula con mayor ROI estimado es {best_roi_movie['film_title']} con ROI {best_roi_movie['roi']:.2f}.",
        "business_reading": "Las peliculas clasicas pueden destacar proporcionalmente aunque las nuevas tengan mayor taquilla absoluta.",
    },
    {
        "area": "Gobernanza",
        "finding": "Existen nulos relevantes en campos fisicos, economicos y demograficos.",
        "business_reading": "Las decisiones deben combinar datos cuantitativos con criterio editorial y de negocio.",
    },
])

display(eda_conclusions)

,area,finding,business_reading
0,Audiencia,La pelicula mas vista es Episode V - The Empir...,Es una candidata fuerte para campanas de alto ...
1,Preferencia,La pelicula con mejor ranking medio es Episode...,La preferencia declarada no siempre coincide c...
2,Personajes,El personaje con mayor indice de merchandising...,Conviene priorizar personajes con alta afinida...
3,Representacion interna,La especie mas frecuente en characters es Huma...,El universo esta concentrado en ciertos grupos...
4,Negocio,La pelicula con mayor taquilla mundial es The ...,El rendimiento comercial complementa la lectur...
5,Rentabilidad,La pelicula con mayor ROI estimado es A New Ho...,Las peliculas clasicas pueden destacar proporc...
6,Gobernanza,"Existen nulos relevantes en campos fisicos, ec...",Las decisiones deben combinar datos cuantitati...


In [36]:
# EXPORTACION DE TABLAS RESUMEN DEL EDA

eda_outputs = {
    "eda_survey_kpis.csv": survey_kpis,
    "eda_movie_views_summary.csv": movie_views_summary,
    "eda_movie_rank_summary.csv": movie_rank_summary,
    "eda_film_business_summary.csv": film_business_summary,
    "eda_movie_commercial_audience_summary.csv": movie_commercial_audience_summary,
    "eda_character_opinion_summary.csv": character_opinion_summary,
    "eda_fan_by_age.csv": fan_by_age,
    "eda_fan_by_gender.csv": fan_by_gender,
    "eda_universe_overview.csv": universe_overview,
    "eda_character_species_summary.csv": character_species_summary,
    "eda_character_gender_summary.csv": character_gender_summary,
    "eda_character_homeworld_summary.csv": character_homeworld_summary,
    "eda_planet_business_summary.csv": planet_business_summary,
    "eda_starship_business_summary.csv": starship_business_summary,
    "eda_weapon_business_summary.csv": weapon_business_summary,
    "eda_quote_character_summary.csv": quote_character_summary,
    "eda_governance_missing_top.csv": governance_missing_top,
    "eda_survey_sample_bias.csv": survey_sample_bias,
    "eda_character_merchandising_opportunities.csv": character_opportunities,
    "eda_movie_opportunities.csv": movie_opportunities,
    "eda_conclusions.csv": eda_conclusions,
}

for filename, df in eda_outputs.items():
    df.to_csv(PROCESSED_DIR / filename, index=False)

print("Tablas de EDA exportadas:")
for filename in sorted(eda_outputs):
    print("-", filename)

Tablas de EDA exportadas:
- eda_character_gender_summary.csv
- eda_character_homeworld_summary.csv
- eda_character_merchandising_opportunities.csv
- eda_character_opinion_summary.csv
- eda_character_species_summary.csv
- eda_conclusions.csv
- eda_fan_by_age.csv
- eda_fan_by_gender.csv
- eda_film_business_summary.csv
- eda_governance_missing_top.csv
- eda_movie_commercial_audience_summary.csv
- eda_movie_opportunities.csv
- eda_movie_rank_summary.csv
- eda_movie_views_summary.csv
- eda_planet_business_summary.csv
- eda_quote_character_summary.csv
- eda_starship_business_summary.csv
- eda_survey_kpis.csv
- eda_survey_sample_bias.csv
- eda_universe_overview.csv
- eda_weapon_business_summary.csv
